In [1]:
%%capture
!pip install wikipedia-api
!pip install country-converter

In [2]:
import requests
import wikipediaapi
from bs4 import BeautifulSoup
import json
import os
from collections import deque
import time
import random
import pandas as pd
import numpy as np
import matplotlib as plt
from collections import Counter
import country_converter as coco
from tqdm import tqdm

In [3]:
#import wikipediaapi
wiki = wikipediaapi.Wikipedia(
    user_agent='MysterySearchEngine/1.0 (contact: your@email.com)',
    language='en',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

In [4]:
def filter_dataframe(data_frame, allowed=None, unallowed=None, contains=None):
    """
    Filters the dataframe based on allowed and unallowed values.
    Example params: allowed={'Action': ['Click', 'View']}, unallowed={'Depth': [0]}
    """
    filtered_df = data_frame.copy()

    # Process "Allowed" values (Must be in the list)
    if allowed:
        for column, values in allowed.items():
            if column in filtered_df.columns:
                filtered_df = filtered_df[filtered_df[column].isin(values)]

    # Process "Unallowed" values (Must NOT be in the list)
    if unallowed:
        for column, values in unallowed.items():
            if column in filtered_df.columns:
                filtered_df = filtered_df[~filtered_df[column].isin(values)]

    # Substring Match: Include (Case insensitive)
    if contains:
        for column, substring in contains.items():
            if column in filtered_df.columns:
                # We convert to string in case the column is numeric (like Depth)
                # na=False prevents errors if there are empty cells
                mask = filtered_df[column].astype(str).str.contains(substring, case=False, na=False)
                filtered_df = filtered_df[mask]

    return filtered_df

In [5]:
def save_progress():
    """Saves all data structures to disk."""
    data = {
        "pages_db": pages_db,
        "crawl_errors": crawl_errors,
        "wikipedia_queue": list(wikipedia_queue),
        "external_links_queue": list(external_links_queue),
        "visited_pages": list(visited_pages),
        "overflow":list(overflow),
        "manual_inspection_bucket": list(manual_inspection_bucket),
        "pages_with_tables": list(pages_with_tables),
        "footprint":list(footprint),
        "see_also_links":list(see_also_title)
    }
    with open(f"{SAVEFILE_DIR}crawler_checkpoint.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print("--- Progress Saved Successfully ---")

def load_progress():
    """Loads progress if a checkpoint exists."""
    global pages_db, crawl_errors, wikipedia_queue, external_links_queue, visited_pages, manual_inspection_bucket,see_also_title,pages_with_tables,overflow,footprint
    if os.path.exists(f"{SAVEFILE_DIR}crawler_checkpoint.json"):
        with open(f"{SAVEFILE_DIR}crawler_checkpoint.json", "r", encoding="utf-8") as f:
            data = json.load(f)
            pages_db = data.get("pages_db", {})
            crawl_errors = data.get("crawl_errors", {})
            wikipedia_queue = deque(data.get("wikipedia_queue", []))
            external_links_queue = set(data.get("external_links_queue", []))
            visited_pages = set(data.get("visited_pages", []))
            manual_inspection_bucket = set(data.get("manual_inspection_bucket", []))
            see_also_title = set(data.get("see_also_links",[]))
            pages_with_tables = list(data.get("pages_with_tables",[]))
            overflow = set(data.get("overflow",[]))
            footprint = list(data.get("footprint",[]))
        print("--- Progress Loaded from Checkpoint ---")

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Experiment

In [ ]:
import requests
import json
from bs4 import BeautifulSoup

def record_page_tables_as_json(title):
    headers = {'User-Agent': 'MysteryBotName (email@example.com)'}
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    content = soup.find('div', class_='mw-parser-output')
    if not content: return []

    scraped_tables = []
    current_hierarchy = {2: "Introduction", 3: None, 4: None, 5: None, 6: None}

    for tag in content.find_all(['h2', 'h3', 'h4', 'h5', 'h6', 'table']):
        if tag.name in ['h2', 'h3', 'h4', 'h5', 'h6']:
            level = int(tag.name[1])
            current_hierarchy[level] = tag.get_text(strip=True).replace('[edit]', '')
            for i in range(level + 1, 7): current_hierarchy[i] = None

        elif tag.name == 'table' and 'wikitable' in tag.get('class', []):
            caption_tag = tag.find('caption')
            table_obj = {
                "section_hierarchy": [current_hierarchy[i] for i in range(2, 7) if current_hierarchy[i]],
                "caption": caption_tag.get_text(strip=True) if caption_tag else None,
                "rows": []
            }

            for tr in tag.find_all('tr'):
                row_cells = []
                for cell in tr.find_all(['td', 'th']):
                    row_cells.append({
                        "text": cell.get_text(strip=True),
                        "is_header": cell.name == 'th',
                        "rowspan": int(cell.get('rowspan', 1)),
                        "colspan": int(cell.get('colspan', 1))
                    })
                if row_cells:
                    table_obj["rows"].append(row_cells)

            scraped_tables.append(table_obj)

    return scraped_tables

# Example Usage
data = record_page_tables_as_json("Dole Air Race")
print(json.dumps(data[0], indent=2))

{
  "section_hierarchy": [
    "The Dole prize",
    "Contestants draw positions"
  ],
  "caption": "Official and unofficial entrants[16]and starting draw[17][12]:\u200a183\u2013184",
  "rows": [
    [
      {
        "text": "Entrant",
        "is_header": true,
        "rowspan": 1,
        "colspan": 4
      },
      {
        "text": "Starting Position",
        "is_header": true,
        "rowspan": 1,
        "colspan": 2
      }
    ],
    [
      {
        "text": "Official / Unofficial",
        "is_header": true,
        "rowspan": 1,
        "colspan": 1
      },
      {
        "text": "Pilot",
        "is_header": true,
        "rowspan": 1,
        "colspan": 1
      },
      {
        "text": "Navigator",
        "is_header": true,
        "rowspan": 1,
        "colspan": 1
      },
      {
        "text": "From",
        "is_header": true,
        "rowspan": 1,
        "colspan": 1
      },
      {
        "text": "Drawn",
        "is_header": true,
        "rowspan": 1,

In [ ]:
record_page_tables_as_json("Python_(programming_language)")

[{'section_hierarchy': ['Hostages'],
  'caption': 'Nationalities and status of the hostages[23][24][3]',
  'rows': [[{'text': 'Country', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Number', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Status', 'is_header': True, 'rowspan': 1, 'colspan': 1}],
   [{'text': 'United States', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': '1', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': 'Killed', 'is_header': False, 'rowspan': 1, 'colspan': 1}],
   [{'text': 'Yemen', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': '7', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': '6 rescued, 1 unknown',
     'is_header': False,
     'rowspan': 1,
     'colspan': 1}],
   [{'text': 'Saudi Arabia', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': '1', 'is_header': False, 'rowspan': 1, 'colspan': 1},
    {'text': 'Rescued', 'is_header': False, 'rowspan': 1, 'colspan':

In [ ]:
record_page_tables_as_json("2014_hostage_rescue_operations_in_Yemen")

In [ ]:
a = record_page_tables_as_json("Dole_Air_Race")
print(len(a))
a

2


[{'section_hierarchy': ['The Dole prize', 'Contestants draw positions'],
  'rows': [[{'text': 'Entrant', 'is_header': True, 'rowspan': 1, 'colspan': 4},
    {'text': 'Starting Position',
     'is_header': True,
     'rowspan': 1,
     'colspan': 2}],
   [{'text': 'Official / Unofficial',
     'is_header': True,
     'rowspan': 1,
     'colspan': 1},
    {'text': 'Pilot', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Navigator', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'From', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Drawn', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Mills/Oakland', 'is_header': True, 'rowspan': 1, 'colspan': 1}],
   [{'text': 'O', 'is_header': True, 'rowspan': 1, 'colspan': 1},
    {'text': 'Arthur C. Goebel',
     'is_header': False,
     'rowspan': 1,
     'colspan': 2},
    {'text': 'Santa Monica, CA',
     'is_header': False,
     'rowspan': 1,
     'colspan': 1},
    {'text': '9', 'is_head

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_img(title):
    results = [] # Stores: {"url": ..., "caption": ...}
    errors = []
    seen_urls = set()

    headers = {
        'User-Agent': 'MysteryBot/1.0 (myemail@example.com)',
        'Referer': 'https://www.google.com/'
    }

    try:
        # 1. Fetch the page
        formatted_title = title.replace(' ', '_')
        url = f"https://en.wikipedia.org/wiki/{formatted_title}"
        response = requests.get(url, headers=headers)

        if response.status_code in [403, 429]:
            raise PermissionError(f"Banned: Status {response.status_code}")

        soup = BeautifulSoup(response.content, 'html.parser')

        # 2. Focus only on article content to keep order and avoid UI icons
        content = soup.find(id="mw-content-text")
        if not content:
            return [], []

        # 3. Find all potential image "wrappers" in document order
        # Wikipedia uses:
        # - .infobox (Main top image)
        # - .thumb (Standard floating images)
        # - .figure (Modern HTML5 image tags)
        # - .gallerybox (Images inside a gallery section)
        wrappers = content.find_all(['table', 'div', 'figure', 'li'],
                                   class_=['infobox', 'thumb', 'thumbinner', 'gallerybox', 'mw-default-size'])

        for wrapper in wrappers:
            img_tag = wrapper.find('img')
            if not img_tag:
                continue

            # Skip small icons/pencils
            width = img_tag.get('width', '0')
            if not (width.isdigit() and int(width) > 50):
                continue

            # Construct URL
            src = img_tag.get('src')
            if not src: continue
            full_url = "https:" + src if src.startswith("//") else src

            # Deduplicate (some wrappers are nested)
            if full_url in seen_urls:
                continue

            # 4. Extract Caption based on container type
            caption = ""

            # Type A: Infobox (Caption is usually in the next <div> or a <tr>)
            if 'infobox' in wrapper.get('class', []):
                # Look for a caption div specifically inside the infobox
                cap_div = wrapper.find(class_='infobox-caption')
                if cap_div:
                    caption = cap_div.get_text(strip=True)
                else:
                    # Fallback: check if the next sibling cell has text
                    parent_td = img_tag.find_parent('td')
                    if parent_td:
                        next_div = parent_td.find('div')
                        if next_div and next_div != img_tag:
                            caption = next_div.get_text(strip=True)

            # Type B: Standard Thumbnail or Gallery
            elif any(cls in wrapper.get('class', []) for cls in ['thumb', 'thumbinner', 'gallerybox']):
                cap_div = wrapper.find(class_=['thumbcaption', 'gallerytext'])
                if cap_div:
                    # Remove the "magnify" icon text often found in thumbcaptions
                    for magnify in cap_div.find_all(class_='magnify'):
                        magnify.decompose()
                    caption = cap_div.get_text(strip=True)

            # Type C: HTML5 Figure
            elif wrapper.name == 'figure':
                cap_tag = wrapper.find('figcaption')
                if cap_tag:
                    caption = cap_tag.get_text(strip=True)

            results.append({
                "url": full_url,
                "caption": caption
            })
            seen_urls.add(full_url)

    except PermissionError:
        raise
    except Exception as e:
        errors.append({"type": "scraping_error", "msg": str(e)})

    return results, errors

In [ ]:
get_img("Python_(programming_language)")

([{'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c3/Python-logo-notext.svg/250px-Python-logo-notext.svg.png',
   'caption': ''},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e0/Python_3.13_Standrd_Type_Hierarchy-en.svg/250px-Python_3.13_Standrd_Type_Hierarchy-en.svg.png',
   'caption': 'The standard type hierarchy in Python\xa03'}],
 [])

In [ ]:
get_img("Dole_Air_Race")

([{'url': 'https://upload.wikimedia.org/wikipedia/commons/a/a1/DoleAirR1927.gif',
   'caption': 'Dole Air Race movie reel (Prelinger Archives)'},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/7/70/James_Drummond_Dole.jpg/250px-James_Drummond_Dole.jpg',
   'caption': 'James D Dole'},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Travel_Air_5000_City_of_Oakland_after_running_out_of_fuel_over_Hawaii.jpg/250px-Travel_Air_5000_City_of_Oakland_after_running_out_of_fuel_over_Hawaii.jpg',
   'caption': 'City of Oaklandafter crash landing onMolokai'},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b4/Matson_Building_%26_Annex_%28San_Francisco%29.jpg/250px-Matson_Building_%26_Annex_%28San_Francisco%29.jpg',
   'caption': 'The draw started at theMatson Building.'},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/10/Dole_Air_Race_Buhl_CA-5_Air_Sedan_NX2915_Miss_Doran.jpg/250px-Dole_Air_Race_Buhl_CA-5_Air_Sedan_NX2915_Mis

In [ ]:
get_img("2014_hostage_rescue_operations_in_Yemen")

([{'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/f/f7/Hagel_Press_Conference_December_6_2014.jpg/330px-Hagel_Press_Conference_December_6_2014.jpg',
   'caption': ''},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/Yemen_relief_location_map.jpg/250px-Yemen_relief_location_map.jpg',
   'caption': 'Location of Dafaar inYemen.'}],
 [])

In [ ]:
get_img("Disappearance_of_Luke_Durbin")

([{'url': 'https://upload.wikimedia.org/wikipedia/en/thumb/f/fb/Luke_Durbin.jpg/250px-Luke_Durbin.jpg',
   'caption': 'Photograph of Luke widely circulated in the wake of his disappearance and used for his missing poster by the charityMissing People.'},
  {'url': 'https://maps.wikimedia.org/img/osm-intl,16,52.055423,1.1549127,400x300.png?lang=en',
   'caption': '104:00 Zebra crossing to bus station, Dogs Head Street location of last confirmed sighting of Durbin'},
  {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Bus_Station_-_geograph.org.uk_-_2849286.jpg/250px-Bus_Station_-_geograph.org.uk_-_2849286.jpg',
   'caption': 'Zebra crossing to bus station, location of last confirmed sighting of Durbin'}],
 [])

In [ ]:
get_img("Jack_the_Ripper")[0]

[{'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a1/JacktheRipper1888.jpg/250px-JacktheRipper1888.jpg',
  'caption': '"With the Vigilance Committee in the East End: A Suspicious Character" fromThe Illustrated London News,13 October1888'},
 {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/6/68/JacktheRipperWhitechapelcirca1890.jpg/250px-JacktheRipperWhitechapelcirca1890.jpg',
  'caption': 'Women and children congregate in front of one of the Whitechapelcommon lodging-housesclose to where Jack the Ripper murdered two of his victims.[1]'},
 {'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/39/Whitechapel_Spitalfields_7_murders.JPG/250px-Whitechapel_Spitalfields_7_murders.JPG',
  'caption': "The sites of the first seven Whitechapel murders\xa0–Osborn Street(centre right),George Yard(centre left),Hanbury Street(top),Buck's Row(far right),Berner Street(bottom right),Mitre Square(bottom left), andDorset Street(middle left)"},
 {'url': 'https://upload

## Main Wikipedia Crawler

In [ ]:
TEXT_LIMIT = None  # Set to a number (e.g., 500) or None for full text
WAIT_RANGE = (1, 3)
BATCH_SIZE = 4
MAX_DEPTH = 2
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"

In [ ]:
# Main Databases
pages_db = {}
# Error tracking: stores {title: {"type": "image|metadata|infobox", "error": "msg"}}
crawl_errors = {}

# Queues & Tracking
wikipedia_queue = deque([("Category:Unsolved crimes",0)])
external_links_queue = set()
visited_pages = set()
manual_inspection_bucket = set()
footprint = []
overflow = set()

pages_with_tables = []  # Stores (Title, URL)
see_also_title = set()     # Stores potential new Case Titles

In [ ]:
def record_page_tables(title):
    headers = {'User-Agent': 'MysteryBotName (email@example.com)'}
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    scraped_tables = []

    for table in soup.find_all('table', class_='wikitable'):
        # Identify section
        header_tag = table.find_previous(['h2', 'h3', 'h4', 'h5', 'h6'])
        section = header_tag.get_text(strip=True).replace('[edit]', '') if header_tag else "Introduction"

        # Extract Headers
        headers_list = [th.get_text(strip=True) for th in table.find_all('th')]

        # Extract Rows
        rows_data = []
        for tr in table.find_all('tr')[1:]: # Skip header row
            cells = tr.find_all(['td', 'th'])
            if cells:
                rows_data.append([cell.get_text(strip=True) for cell in cells])

        scraped_tables.append({
            "section": section,
            "headers": headers_list,
            "data": rows_data
        })

    return scraped_tables


def get_table_links(title, col_name, allowed_sections=None):
    """
    allowed_sections: list of strings or None to get all tables.
    col_name: the exact name of the column 'X'
    """
    headers = {'User-Agent': 'BotName (email@example.com)'}
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    found_links = []

    for table in soup.find_all('table', class_='wikitable'):
        # 1. Check Section Permission
        header_tag = table.find_previous(['h2', 'h3', 'h4', 'h5', 'h6'])
        section = header_tag.get_text(strip=True).replace('[edit]', '') if header_tag else "Introduction"

        if allowed_sections is not None and section not in allowed_sections:
            continue

        # 2. Find Column Index
        ths = [th.get_text(strip=True) for th in table.find_all('th')]
        try:
            col_index = ths.index(col_name)
        except ValueError:
            continue # Column 'X' not in this table

        # 3. Extract Links from that column
        for tr in table.find_all('tr'):
            cells = tr.find_all(['td', 'th'])
            if len(cells) > col_index:
                target_cell = cells[col_index]
                # Find all links in this specific cell
                links = target_cell.find_all('a', href=True)
                for l in links:
                    href = l['href']
                    if href.startswith('/wiki/'):
                        found_links.append(f"https://en.wikipedia.org{href}")
                    elif href.startswith('http'):
                        found_links.append(href)

    return list(set(found_links)) # Return unique links

In [ ]:
def get_advanced_data(title):
    images = []
    infobox_data = {}
    errors = []
    last_updated_text = ""

    # Same headers you used for BeautifulSoup
    headers = {
        'User-Agent': 'MysteryBot/1.0 (myemail@example.com)',
        'Referer': 'https://www.google.com/'
    }

    try:
        url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
        response = requests.get(url, headers=headers)

        if response.status_code in [403, 429]:
            raise PermissionError(f"Banned: Status {response.status_code}")

        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. GET INFOBOX DATA
        infobox = soup.find("table", {"class": "infobox"})
        if infobox:
            for row in infobox.find_all("tr"):
                label = row.find("th")
                value = row.find("td")
                if label and value:
                    infobox_data[label.get_text(strip=True)] = value.get_text(strip=True)

        # 2. GET IMAGES FROM HTML (Bypassing the API)
        # We look for images in the infobox or the main content area
        # Filtering out tiny icons (less than 50px)
        img_tags = soup.find_all('img')
        for img in img_tags:
            src = img.get('src')
            if src:
                # Filter out small icons like edit pencils or UI elements
                width = img.get('width', '0')
                if width.isdigit() and int(width) > 50:
                    # Convert //upload... to https://upload...
                    full_url = "https:" + src if src.startswith("//") else src
                    images.append(full_url)

        # 4. Last Modified date
        footer_mod = soup.find("li", {"id": "footer-info-lastmod"})
        last_updated_text = footer_mod.get_text(strip=True) if footer_mod else "Unknown"

    except PermissionError:
        raise # Re-raise to trigger the main loop's "save and stop" logic
        print("permission error!")
    except Exception as e:
        errors.append({"type": "scraping_error", "msg": str(e)})
        print(f"scraping error:{str(e)}")

    return last_updated_text,images, infobox_data, errors

In [ ]:
def get_sections_tree(sections, text_limit=None):
    tree = []
    for s in sections:
        # Truncate text if limit is set
        content_text = s.text if text_limit is None else s.text[:text_limit]

        section_node = {
            "title": s.title,
            "content": [content_text] # List starts with the section's text
        }

        # If there are subsections, recurse and add them to the same content list
        if s.sections:
            section_node["content"].append(get_sections_tree(s.sections, text_limit))

        tree.append(section_node)
    return tree

In [ ]:
def run_crawler(limit=5, force_visit=False):
    try:
        processed = 0
        open_category = 0
        BATCH_CAT = 10
        while wikipedia_queue and processed < limit:
          current_title, depth, origin_title_raw = wikipedia_queue.popleft()
          ordinary = True
          origin_title = origin_title_raw
          if origin_title_raw[0] == "$":
              origin_title = origin_title_raw[1:]
              ordinary = False

          if not force_visit:
            if current_title in visited_pages:
                print(f"already visited {current_title}")
                continue

          if depth > MAX_DEPTH:
              overflow.add(current_title)
              footprint.append([current_title,origin_title,"overflow",depth])
              print(f"overflow {current_title}")
              continue

          page = wiki.page(current_title)
          if not page.exists():
              print(f"page {current_title} don't exist")
              continue

          print(f"Crawling: {page.title}")
          visited_pages.add(page.title)
          footprint.append([current_title,origin_title,"crawl",depth])

          #time.sleep(DELAY_SECONDS)

          # Handle CATEGORY
          if page.ns == wikipediaapi.Namespace.CATEGORY:
              for member in page.categorymembers.values():
                  wikipedia_queue.append((member.title,depth+1,current_title))
                  footprint.append([member.title,current_title,"category",depth+1])
              open_category += 1
              if open_category > BATCH_CAT:
                  time.sleep(random.uniform(*WAIT_RANGE))
                  open_category = 0

          # Handle LIST
          elif ordinary and (page.title.startswith("List of ") or page.title.startswith("Lists of ")):
              open_category = 0
              #check_title = page.title.lower()
              #if ("unsolved" in check_title):
              #    for link_title in page.backlinks.keys():
              #        wikipedia_queue.append((link_title, depth+1,current_title))
              #        footprint.append([link_title,current_title,"list unsolved",depth+1])
              #else:
              manual_inspection_bucket.add(page.title)
              footprint.append([current_title,origin_title,"list_manual_inspect",depth+1])

          # Handle ARTICLE
          else:
              open_category = 0
              # 1. Scrape Advanced Data (Images & Infobox)
              last_updated_text, imgs, infobox, errors = get_advanced_data(page.title)

              #print('passed advance')

              # 2. Log Errors if any
              if errors:
                  crawl_errors[page.title] = errors

              # 3. Store Article Data
              pages_db[page.title] = {
                  "origin_title":origin_title,
                  "url": page.fullurl,
                  "last_update_text":last_updated_text,
                  "summary": page.summary if TEXT_LIMIT is None else page.summary[:TEXT_LIMIT],
                  "infobox": infobox,
                  "sections_tree":  get_sections_tree(page.sections, TEXT_LIMIT),  #{s.title: s.text if TEXT_LIMIT is None else s.text[:TEXT_LIMIT] for s in page.sections }
                  "table":record_page_tables(page.title),
                  "categories": list(page.categories.keys())
              }
              #print('passed page db')

              # 4. Handle "See Also" links
              see_also_sec = [s for s in page.sections if s.title.lower() == "see also"]
              if see_also_sec:
                  # Add links mentioned in the 'See Also' section
                  # We filter page.links to only those appearing in that section's text
                  for link_title in page.links.keys():
                      if link_title in see_also_sec[0].text:
                          #wikipedia_queue.append(link_title)
                          see_also_title.add(link_title)
                          footprint.append([link_title,current_title,"see also",depth+1])
                          #manual_inspection_bucket.add(link_title)
              processed += 1
              if processed % BATCH_SIZE == 0:
                  time.sleep(random.uniform(*WAIT_RANGE))

              if processed >= limit:
                  print("hit limit :)")
                  break
                  #processed += 1
        print("Finished while loop")
    except PermissionError as e:
        print(f"!!! CRITICAL: Stopped because of ban: {e}")
    except Exception as e:
        print(f"!!! Unexpected Error: {e}")
    finally:
        # This ALWAYS runs, ensuring your data is not lost
        save_progress()
        print("Crawler stopped. Progress has been secured.")


In [ ]:
TEXT_LIMIT = None  # Set to a number (e.g., 500) or None for full text
WAIT_RANGE = (1, 3)
BATCH_SIZE = 4
MAX_DEPTH = 3
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"
# Main Databases
pages_db = {}
crawl_errors = {} # Error tracking: stores {title: {"type": "image|metadata|infobox", "error": "msg"}}

# Queues & Tracking
wikipedia_queue = deque([("Category:Unsolved crimes",0,"seed")])
external_links_queue = set()
visited_pages = set()
manual_inspection_bucket = set()
overflow = set()
footprint = []

pages_with_tables = []  # Stores (Title, URL)
see_also_title = set()     # Stores potential new Case Titles

# --- EXECUTION ---
#wikipedia_queue = deque([("Category:Missing_aircraft",0,"seed"),("Category:Missing_ships",0,"seed")])
wikipedia_queue = deque([("Category:Unexplained_disappearances",0,"seed")])
MAX_DEPTH = 2

In [ ]:
#wikipedia_queue = deque([("Cold case",0,"seed")])
#SAVEFILE_DIR = ""
load_progress() # Resume if possible
wikipedia_queue = deque([("Category:Unexplained_disappearances",0,"seed")])
run_crawler(limit=10000) # Adjust limit as needed
save_progress() # Save results

Crawling: Croye Pithey
Crawling: Arthur Piver
Crawling: Rosendo Porlier y Asteguieta
already visited Walter Powell (MP for Malmesbury)
Crawling: Tyrone Power (Irish actor)
Crawling: John James Powers
already visited Preston (1798 EIC ship)
Crawling: Günther Prien
Crawling: Angus Primrose
Crawling: Cecil Pugh
Crawling: Douglas Pyne
Crawling: Lynne C. Quiggle
already visited 1969 RAF Mildenhall C-130 theft
Crawling: Rafael Villaverde
Crawling: Alfred Raper
Crawling: Henry Hope Reed
Crawling: Alonso de Reinoso
Crawling: Kemal Reis
Crawling: David Thomas Richardson
Crawling: HMS Richmond helicopter crash
Crawling: Ridderschap van Holland (1682)
Crawling: Esmond Romilly
Crawling: Gerard Broadmead Roope
Crawling: Jürgen von Rosenstiel
Crawling: Inigo Ross
already visited Holly Roth
Crawling: Gerry Roufs
Crawling: Michael Rudolph
Crawling: Marc Armand Ruffer
Crawling: Boris Safonov
Crawling: Joseph Samuel
Crawling: Disappearance of ARA San Juan
Crawling: Giuseppe Sartorio
Crawling: Auzella Sa

In [ ]:
len(pages_db)

In [ ]:
footprint

## Cleaning

In [ ]:
load_progress()

--- Progress Loaded from Checkpoint ---


In [ ]:
cleaning_pages_db = pages_db.copy()

In [ ]:
len(pages_db)

2474

In [ ]:
df_footprint = pd.DataFrame(footprint, columns=['Curr Title', 'Origin', 'Action', 'Depth'])

In [ ]:
df_footprint['Action'].unique()

array(['crawl', 'category', 'see also', 'list unsolved',
       'list not unsolved-manual_inspect', 'overflow'], dtype=object)

In [ ]:
df_crawled = df_footprint[df_footprint['Action']=='crawl']

In [ ]:
df_crawled['Origin'].unique()

array(['seed', 'Category:Unsolved crimes',
       'Category:Unsolved crimes by continent',
       'Category:Unsolved crimes by country', 'Category:Unsolved arsons',
       'Category:Unidentified criminals', 'Category:Unsolved murders',
       'Category:Terrorist incidents by unknown perpetrators',
       'Category:Works about unsolved crimes',
       'Category:Missing person cases by continent',
       'Category:Unsolved murders by continent',
       'Category:Unsolved crimes in Europe',
       'Category:Unsolved murders by country',
       'Category:Unsolved crimes in Angola',
       'Category:Unsolved crimes in Australia',
       'Category:Unsolved crimes in Belize',
       'Category:Unsolved crimes in Brazil',
       'Category:Unsolved crimes in Cambodia',
       'Category:Unsolved crimes in Canada',
       'Category:Unsolved crimes in China',
       'Category:Unsolved crimes in Costa Rica',
       'Category:Unsolved crimes in France',
       'Category:Unsolved crimes in Germany',
 

In [ ]:
len(cleaning_pages_db)

2114

In [ ]:
erased_title = []

In [ ]:
erased_title[-10:]

['Monumento a los Niños Héroes (Guadalajara)',
 'Paine Memorial',
 'The Perils of Penelope',
 'Pig Girl',
 'Runaway Train (Soul Asylum song)',
 'Sabrina (comics)',
 'Something Is Killing the Children',
 'The Vanished (podcast)',
 'Walking With Our Sisters',
 "Zahra's Paradise"]

In [ ]:
banned_cat = ['Category:Works about the Black Dahlia case','Category:Works about Tupac Shakur','Category:Works about unsolved murders in the United States',
              'Category:Non-fiction books about unsolved murders in the United States','Category:2007 Samjhauta Express bombings','Category:Formerly missing people',
              'Category:Formerly missing people by nationality','Category:Deaths in the Titan submersible implosion','Category:Missing people identified through genetic genealogy',
              'Category:Lost specimens','Category:Bermuda Triangle in fiction','Category:Works set in the Bermuda Triangle','Category:Films set in the Bermuda Triangle',
              'Category:Novels set in the Bermuda Triangle', 'Category:Television episodes set in the Bermuda Triangle', 'Category:Television shows set in the Bermuda Triangle',
              'Category:Video games set in the Bermuda Triangle','Category:Books about the Bermuda Triangle','Category:Mary Celeste','Category:Lost astronomical objects',
              'Category:Lost comets','Category:Lost minor planets','Category:Lost moons','Category:Recovered astronomical objects','Category:Works about missing people',
              'Category:Films about missing people','Category:Novels about missing people','Category:Short stories about missing people','Category:Television series about missing people',
              'Category:Video games about missing people','Category:Documentaries about missing people','Category:Search and rescue','Category:Search and rescue helicopters',
              'Category:Search and rescue dogs','Category:Search patterns','Category:Castaways','Category:Fictional castaways','Category:Fiction about castaways',
              'Category:Fictional characters incorrectly presumed dead','Category:Murder convictions without a body'
              ]

In [ ]:
#df_crawled['Origin']
df_works_unsolved_crimes = filter_dataframe(df_footprint, {'Origin':['Category:People lost at sea']})
df_works_unsolved_crimes

,Curr Title,Origin,Action,Depth
12786,List of people who disappeared mysteriously at...,Category:People lost at sea,category,3
12787,1957 Pacific Ocean Boeing C-97 disappearance,Category:People lost at sea,category,3
12788,2008 disappearance of CIA operatives in the So...,Category:People lost at sea,category,3
12789,2019 English Channel Piper PA-46 crash,Category:People lost at sea,category,3
12790,Aagtekerke (1724),Category:People lost at sea,category,3
...,...,...,...,...
19114,John Yarnall,Category:People lost at sea,crawl,3
19116,Yong Yu Sing No. 18,Category:People lost at sea,crawl,3
19119,John Young (naval officer),Category:People lost at sea,crawl,3
19121,Lamont Young,Category:People lost at sea,crawl,3


In [ ]:
page_banned = list(df_works_unsolved_crimes['Curr Title'].unique())
for title in page_banned:
  if 'Category:' in title:
    print(title)

Category:Fictional characters incorrectly presumed dead
Category:Murder convictions without a body


In [ ]:
page_banned = list(df_works_unsolved_crimes['Curr Title'].unique())
for title in page_banned:
  print(title)
  if 'Category:' in title:
    print("category")
  else:
    _ = cleaning_pages_db.pop(title,None)
    erased_title.append(title)

In [ ]:
TEXT_LIMIT = None  # Set to a number (e.g., 500) or None for full text
WAIT_RANGE = (1, 3)
BATCH_SIZE = 4
MAX_DEPTH = 3
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"
# Main Databases
pages_db = {}
crawl_errors = {} # Error tracking: stores {title: {"type": "image|metadata|infobox", "error": "msg"}}

# Queues & Tracking
wikipedia_queue = deque([("Category:Unsolved crimes",0,"seed")])
external_links_queue = set()
visited_pages = set()
manual_inspection_bucket = set()
overflow = set()
footprint = []

pages_with_tables = []  # Stores (Title, URL)
see_also_title = set()     # Stores potential new Case Titles

In [ ]:
see_also_title

{'American literature in Spanish',
 'Crime in Georgia (U.S. state)',
 'List of pioneers in computer science',
 '2013 Neo Irakleio Golden Dawn office shooting',
 'Crime in North Carolina',
 'Crime in Western Australia',
 'Kingdom of Galicia–Volhynia',
 'Trial of Yolanda Saldívar',
 'James Randi',
 'Lyle Stevik',
 'Ignacio Ellacuría',
 'Death of David Glenn Lewis',
 'List of American independent films',
 'Abahlali baseMjondolo',
 'Significant acts of violence against LGBT people',
 'Domestic terrorism in the United States',
 'List of people from Massachusetts',
 'Tim Vakoc',
 'Lyre',
 'Pavel Sheremet',
 'Terrorism in the Philippines',
 'Francesca Mambro',
 'Misa Campesina Nicaragüense',
 'Sydney gangland war',
 'Josephus',
 'Frederick Forsyth',
 'Extrajudicial killings and forced disappearances in the Philippines',
 'Prostitution in Germany',
 'Memoir',
 'Ciudad Juárez',
 'Sri Lanka',
 'Yvan Leyvraz',
 'List of suffragists and suffragettes',
 'List of missing aircraft',
 'St.Phillip Neri

In [ ]:
df_banned = filter_dataframe(df_footprint, {'Origin':banned_cat})
banned_titles = df_banned['Curr Title'].unique()


In [ ]:
cleaned_manual_inspection_bucket = [item for item in manual_inspection_bucket if item not in banned_titles]
cleaned_overflow = [item for item in overflow if item not in banned_titles]
cleaned_see_also_title = [item for item in see_also_title if item not in banned_titles]
cleaned_pages_with_tables = [item for item in pages_with_tables if item[0] not in banned_titles]

In [ ]:
for t in banned_titles:
  k = cleaning_pages_db.pop(t,None)
  if not k is None:
    print(k)

{'origin_title': 'Category:Unexplained_disappearances', 'url': 'https://en.wikipedia.org/wiki/Mary_Celeste', 'last_update_text': 'This page was last edited on 18 May 2026, at 05:04(UTC).', 'summary': 'Mary Celeste (, often erroneously referred to as Marie Celeste,) was a Canadian-built, American-registered, merchant brigantine that was discovered adrift and deserted in the Atlantic Ocean off the Azores on December 4, 1872. \nThe Canadian brigantine Dei Gratia found her in a dishevelled but seaworthy condition under partial sail and with her lifeboat missing. The last entry in her log was dated 10 days earlier. She had left New York City for Genoa on November 7 and was still amply provisioned when found. Her cargo of alcohol was intact, and the captain\'s and crew\'s personal belongings were undisturbed. None of those who had been on board were ever seen or heard from again.\nMary Celeste was built in Spencer\'s Island, Nova Scotia, and launched under British registration as Amazon in 1

In [ ]:
len(cleaned_overflow)

2058

In [ ]:
len(overflow)

3335

In [ ]:
data = {
    "pages_db": cleaning_pages_db,
    "crawl_errors": crawl_errors,
    "wikipedia_queue": list(wikipedia_queue),
    "external_links_queue": list(external_links_queue),
    "visited_pages": list(visited_pages),
    "overflow":list(cleaned_overflow),
    "manual_inspection_bucket": list(cleaned_manual_inspection_bucket),
    "pages_with_tables": list(cleaned_pages_with_tables),
    "footprint":list(footprint),
    "see_also_links":list(cleaned_see_also_title)
}
with open(f"{SAVEFILE_DIR}crawl2_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)
print("--- Cleaning Saved Successfully ---")


--- Cleaning Saved Successfully ---


## Inspection

In [ ]:
load_progress()

In [ ]:
erasing_manualinsp = ['List of Whaling Walls','List of solved missing person cases (1970s)','List of lists of lists','Lists of murders','Lists of assassinations',
                      'List of people who disappeared mysteriously (pre-1910)','List of previously missing aircraft','Lists of people who disappeared']
presented_as_page = ['List of missing treasures','List of Bermuda Triangle incidents','List of destroyed heritage','List of missing aircraft']

In [ ]:
inspected_manual_inspection_bucket = [item for item in manual_inspection_bucket if item not in erasing_manualinsp]
inspected_manual_inspection_bucket = [item for item in inspected_manual_inspection_bucket if item not in presented_as_page]

In [ ]:
inspected_manual_inspection_bucket

['List of people who disappeared mysteriously (1980s)',
 'List of people who disappeared mysteriously (2000–present)',
 'List of people who disappeared mysteriously at sea',
 'List of missing ships',
 'List of people who disappeared mysteriously (1970s)',
 'List of people who disappeared mysteriously (1990s)',
 'List of fugitives from justice who disappeared',
 'List of people who disappeared mysteriously (1910–1970)']

In [ ]:
direct_links_in_table = [
    ['List of missing ships','Ship'],
    ['List of people who disappeared mysteriously (1980s)','Person(s)'],
    ['List of people who disappeared mysteriously (2000–present)','Person(s)'],
    ['List of people who disappeared mysteriously (1970s)','Person(s)'],
    ['List of people who disappeared mysteriously (1990s)','Person(s)'],

]
direct_links_in_table_restrict = [
    ['List_of_fugitives_from_justice_who_disappeared',{"Allowed":['1950s','1960s','1970s','1980s','1990s','2000s','2010s','2020s']},'Person(s)'],
    ['List of people who disappeared mysteriously at sea',{"Allowed":['1970–present']},'Person(s)'],

]

In [ ]:
df_footprint = pd.DataFrame(footprint, columns=['Curr Title', 'Origin', 'Action', 'Depth'])

In [ ]:
df_overflow = df_footprint[df_footprint['Curr Title'].isin(overflow)]
df_overflow = df_overflow[df_overflow['Action']=='overflow']

In [ ]:
len(df_overflow)

2271

In [ ]:
len(overflow)

2058

In [ ]:
df_overflow

,Curr Title,Origin,Action,Depth
9718,Vasco de Ataíde,Category:Missing person cases in Africa,overflow,4
9719,Jay Emmanuel Banda,Category:Missing person cases in Africa,overflow,4
9720,Clint Castleberry,Category:Missing person cases in Africa,overflow,4
9721,Zaida Catalán,Category:Missing person cases in Africa,overflow,4
9724,Douglas Clavering,Category:Missing person cases in Africa,overflow,4
...,...,...,...,...
21556,Category:Enforced disappearances in the Philip...,Category:Enforced disappearances by country,overflow,4
21557,Category:Enforced disappearances in the Soviet...,Category:Enforced disappearances by country,overflow,4
21558,Category:Enforced disappearances in Sri Lanka,Category:Enforced disappearances by country,overflow,4
21559,Category:Enforced disappearances in Thailand,Category:Enforced disappearances by country,overflow,4


In [ ]:
df_overflow['Origin'].unique()

array(['Category:Missing person cases in Africa',
       'Category:Missing person cases in Asia',
       'Category:Missing person cases in Europe',
       'Category:Missing person cases in North America',
       'Category:Missing person cases in Oceania',
       'Category:Missing person cases in South America',
       'Category:Unsolved murders in Africa',
       'Category:Unsolved murders in Asia',
       'Category:Unsolved murders in North America',
       'Category:Unsolved murders in South America',
       'Category:Unidentified murderers by nationality',
       'Category:Unsolved murders in Afghanistan',
       'Category:Unsolved murders in Angola',
       'Category:Unsolved murders in Argentina',
       'Category:Unsolved murders in Australia',
       'Category:Unsolved murders in Austria',
       'Category:Unsolved murders in the Bahamas',
       'Category:Unsolved murders in Bangladesh',
       'Category:Unsolved murders in Belgium',
       'Category:Unsolved murders in Belize'

In [ ]:
origin_okay_overflow = []
for i, row in df_overflow.iterrows():
  if 'unsolved' in row['Origin'].lower() or 'unidentified' in row['Origin'].lower():
    origin_okay_overflow.append(row['Curr Title'])
print(len(origin_okay_overflow))

933


In [ ]:
remaining_overflow = df_overflow[~df_overflow['Curr Title'].isin(origin_okay_overflow)]

In [ ]:
eraising_origin = ['Category:Disappearance of Madeleine McCann','Category:Lists of kidnappings','Category:Intentionally destroyed artworks',
           'Category:Taxa with lost type specimens','Category:Grandmothers of the Plaza de Mayo','Category:Stolen and missing Moon rocks',
           'Category:Nazi-looted art', 'Category:Lost paintings','Category:Reconstructed musical instruments','Category:Lost sculptures',
           'Category:Recovered works of art', 'Category:Mothers of the Plaza de Mayo','Category:Vietnam War POW/MIA activists',
            'Category:Missing people organizations','Category:Hanging Gardens of Babylon','Category:Ark of the Covenant','Category:Lost Fabergé eggs',
            'Category:Lost castles','Category:Disappeared princes'
]
add = ['Disappearance_of_Madeleine_McCann','Malaysia_Airlines_Flight_370']
maybe_add = ['Hanging_Gardens_of_Babylon','Ark of the Covenant']

In [ ]:
remaining_overflow = remaining_overflow[~remaining_overflow['Origin'].isin(eraising_origin)]

In [ ]:
remaining_overflow['Origin'].unique()

array(['Category:Missing person cases in Africa',
       'Category:Missing person cases in Asia',
       'Category:Missing person cases in Europe',
       'Category:Missing person cases in North America',
       'Category:Missing person cases in Oceania',
       'Category:Missing person cases in South America',
       'Category:Malaysia Airlines Flight 370',
       'Category:Missing ships of Australia',
       'Category:Lost sailing vessels', 'Category:Missing submarines',
       'Category:Missing person cases by country of disappearance',
       'Category:Missing children by nationality',
       'Category:Missing Afghan people',
       'Category:Missing American people',
       'Category:Missing Argentine people',
       'Category:Missing Australian people',
       'Category:Missing Austrian people',
       'Category:Missing Bangladeshi people',
       'Category:Missing Belgian people',
       'Category:Missing Brazilian people',
       'Category:Missing British people',
       'Categ

In [ ]:
remaining_overflow_title = list(remaining_overflow['Curr Title'].unique())

In [ ]:
# nned to process again
presented_as_page
direct_links_in_table
direct_links_in_table_restrict
origin_okay_overflow
remaining_overflow_title

False

In [ ]:
data = {
    "presented_as_page":presented_as_page,
    "direct_links_in_table":direct_links_in_table,
    "direct_links_in_table_restrict":direct_links_in_table_restrict,
    "origin_okay_overflow":origin_okay_overflow,
    "remaining_overflow_title":remaining_overflow_title,
}
with open(f"{SAVEFILE_DIR}inspected_next_seed_from2.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)
print("--- Inspection Next Seed Saved Successfully ---")

--- Inspection Next Seed Saved Successfully ---


## Round 2

In [ ]:
filename = "inspected_next_seed_from2.json"
with open(f"{SAVEFILE_DIR}{filename}", "r", encoding="utf-8") as f:
    next_data = json.load(f)

In [ ]:
presented_as_page = next_data['presented_as_page']
direct_links_in_table = next_data['direct_links_in_table']
direct_links_in_table_restrict = next_data['direct_links_in_table_restrict']
origin_okay_overflow = next_data['origin_okay_overflow']
remaining_overflow_title = next_data['remaining_overflow_title']

In [ ]:
load_progress()

In [ ]:
new_queue = []

In [ ]:
df_footprint = pd.DataFrame(footprint, columns=['Curr Title', 'Origin', 'Action', 'Depth'])

In [ ]:
all_title = []
all_title.extend(remaining_overflow_title)
all_title.extend(origin_okay_overflow)
all_title = list(set(all_title))
all_pres_page = []
all_pres_page.extend(presented_as_page)
all_pres_page = list(set(all_pres_page))

In [ ]:
df_title = df_footprint[df_footprint['Curr Title'].isin(all_title)]
df_title = df_title[df_title['Action']=='overflow']
for i, row in df_title.iterrows():
  if row['Origin'].lower()=='cold case':
    continue
  new_queue.append((row['Curr Title'],0,row['Origin']))
print(len(new_queue))

1714


In [ ]:
df_title = df_footprint[df_footprint['Curr Title'].isin(presented_as_page)]
df_title = df_title[df_title['Action']=='list not unsolved-manual_inspect']
for i, row in df_title.iterrows():
  new_queue.append((row['Curr Title'],0,"$"+row['Origin']))
print(len(new_queue))

1718


In [ ]:
new_links = []
prefix = 'https://en.wikipedia.org/wiki/'
for p in direct_links_in_table:
  links = get_table_links(p[0], p[-1], allowed_sections=None)
  print(f"{p[0]}: {len(links)} links")
  cleaned_title = []
  for l in links:
    if l.startswith(prefix):
      cleaned_title.append((l[len(prefix):],0,p[0]))
    else:
      cleaned_title.append((l,0,p[0]))
  new_links.extend(cleaned_title)
print(len(new_links))

List of missing ships: 131 links
List of people who disappeared mysteriously (1980s): 85 links
List of people who disappeared mysteriously (2000–present): 255 links
List of people who disappeared mysteriously (1970s): 99 links
List of people who disappeared mysteriously (1990s): 99 links
669


In [ ]:
for p in direct_links_in_table_restrict:
  links = get_table_links(p[0], p[-1], allowed_sections=p[1]['Allowed'])
  print(f"{p[0]}: {len(links)} links")
  cleaned_title = []
  for l in links:
    if l.startswith(prefix):
      cleaned_title.append((l[len(prefix):],0,p[0]))
    else:
      cleaned_title.append((l,0,p[0]))
  new_links.extend(cleaned_title)
print(len(new_links))

List_of_fugitives_from_justice_who_disappeared: 132 links
List of people who disappeared mysteriously at sea: 45 links
846


In [ ]:
new_queue.extend(new_links)
print(len(new_queue))

2564


In [ ]:
TEXT_LIMIT = None  # Set to a number (e.g., 500) or None for full text
WAIT_RANGE = (1, 3)
BATCH_SIZE = 5
MAX_DEPTH = 2

# Queues & Tracking
#wikipedia_queue = deque([("Category:Unsolved crimes",0,"seed")])
external_links_queue = set()
manual_inspection_bucket = set()
overflow = set()
#footprint = []
#pages_with_tables = []  # Stores (Title, URL)
#see_also_title = set()     # Stores potential new Case Titles

In [ ]:
wikipedia_queue = deque(new_queue)

In [ ]:
run_crawler(limit=10000) # Adjust limit as needed
save_progress() # Save results

Crawling: USS Lynx (1814)
Crawling: German submarine U-703
Crawling: German submarine U-529
Crawling: German submarine U-184
Crawling: Aagtekerke (1724)
Crawling: German submarine U-381
Crawling: Amy (ship)
Crawling: SS Waratah
Crawling: USS Snook (SS-279)
Crawling: SS W.H. Gilcher
Crawling: German submarine U-376
Crawling: General Grant (sailing ship)
Crawling: USS Swordfish (SS-193)
Crawling: HMS York (1660)
Crawling: Hanneke Vrome
Crawling: MS Hans Hedtoft
Crawling: SS Bannockburn
Crawling: German submarine U-420
Crawling: Neustria (ship)
Crawling: SS Java (1865)
Crawling: Lyman D. Foster
Crawling: Bounty (1960 ship)
Crawling: French minesweepers Inkerman and Cerisoles
Crawling: USS Capelin
page ORP_Orze%C5%82_(1938) don't exist
Crawling: USS Runner (SS-275)
Crawling: Disappearance of Mirella Gregori
Crawling: Disappearance of Nyleen Marshall
page Agust%C3%ADn_Feced don't exist
page Ala%C3%ADde_Foppa don't exist
Crawling: Disappearance of Diane Suzuki
Crawling: Disappearance of Mela

In [ ]:
wikipedia_queue

deque([])

In [ ]:
manual_inspection_bucket = set() #no useful next seed

In [ ]:
overflow

set()

In [ ]:
len(pages_db)

5421

In [ ]:
save_progress()

--- Progress Saved Successfully ---


## Final Clean

In [ ]:
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"
load_progress()
len(pages_db)

--- Progress Loaded from Checkpoint ---


5422

### Remove Obvious Page Titles & Origins

In [ ]:
cp_pages_db = pages_db.copy()

In [ ]:
page_title = list(cp_pages_db.keys())
rem_num = 0
for keys in page_title:
  #if keys.startswith("User:") or keys.startswith("User talk:") or keys.startswith("Wikipedia:") or keys.startswith("Wikipedia talk:") or keys.startswith("File:") or '.jpg' in keys:
  if keys.startswith("Talk:"):
    delmaybe = cp_pages_db.pop(keys, None)
    if delmaybe:
      rem_num += 1
print(rem_num)

28


In [ ]:
len(cp_pages_db)

5154

In [ ]:
page_title = list(cp_pages_db.keys())
page_title.sort()
for i in page_title:
  print(i)

11th Panchen Lama controversy
18 July 2012 Damascus bombing
1894 Rock Island railroad wreck
1896 Barcelona Corpus Christi procession bombing
1901 Redpath Mansion murders
1924 KLM Fokker F.III disappearance
1931 Avro Ten Southern Cloud disappearance
1939 City of San Francisco derailment
1945 RAAF Douglas C-47 disappearance
1948 Airborne Transport DC-3 disappearance
1948 Beechcraft Model 18 disappearance
1949 Menarsha synagogue bombing
1950 Douglas C-54D disappearance
1951 Atlantic C-124 disappearance
1953 Skyways Avro York disappearance
1955 RAF Shackleton aircraft disappearance
1956 Atlantic R6D-1 disappearance
1956 B-47 disappearance
1957 Pacific Ocean Boeing C-97 disappearance
1965 Argentine Air Force C-54 disappearance
1965–1966 Columbus murders
1966 theft of the Jules Rimet Trophy
1969 RAF Mildenhall C-130 theft
1972 Harlem mosque incident
1972 Montreal Museum of Fine Arts robbery
1972 Queenstown shooting
1973 Canadian Imperial Bank of Commerce bank robbery
1974 United States Air F

In [ ]:
page_origin = set()
for key, value in cp_pages_db.items():
  page_origin.add(value['origin_title'])
page_origin = list(page_origin)
page_origin.sort()
for i in page_origin:
  print(i)

Category:1490s missing person cases
Category:1500s missing person cases
Category:1570s missing person cases
Category:1580s missing person cases
Category:1590s missing person cases
Category:1620s missing person cases
Category:1670s missing person cases
Category:1700s missing person cases
Category:1730s missing person cases
Category:1790s missing person cases
Category:17th-century missing person cases
Category:1800s missing person cases
Category:1810s missing person cases
Category:1820s missing person cases
Category:1830s missing person cases
Category:1840s missing person cases
Category:1850s missing person cases
Category:1860s missing person cases
Category:1870s missing person cases
Category:1880s missing person cases
Category:1890s missing person cases
Category:1900s missing person cases
Category:1910s missing person cases
Category:1920s missing person cases
Category:1930s missing person cases
Category:1940s missing person cases
Category:1950s missing person cases
Category:1960s missin

In [ ]:
erase_origin = ["Category:Castaways","Category:Highway of Tears","Category:Jack the Ripper","Category:Kidnapped Palestinian people",
                "Category:Lost and extinct musical instruments","Category:Melbourne gangland killings","Category:Moran family",
                "Category:POW/MIA advocacy","Category:Search for Malaysia Airlines Flight 370","Category:Vietnam War POW/MIA issues",
                "Category:Vietnam War POW/MIA issues","Category:Victims of the Melbourne gangland killings","Category:Works about missing people"]

to_add_page = [
    ("Black_Dahlia",0,"Unsolved murders in California"),
              ("Murder_of_Tupac_Shakur",0,"Unsolved murders in Nevada"),
               #("2007_Samjhauta_Express_bombings",0,
               ("Mary Celeste",0,"Unexplained disappearances"),
               ("Disappearance of Don Banfield",0,"Unsolved murders in London"),
               ("Jack the Ripper",0,"Unidentified British serial killers"),
               ("Malaysia Airlines Flight 370",0,"Missing aircraft")]

In [ ]:
rem_keys = []
for key, value in c1_pages_db.items():
  if value['origin_title'] in erase_origin:
    origin = value['origin_title']
    if key == origin[9:]:
      print("spared:"+key)
    else:
      rem_keys.append(keys)
print(len(rem_keys))
for i in rem_keys:
  _ = c1_pages_db.pop(i,None)

spared:Highway of Tears
76


In [ ]:
rem_keys

['Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste',
 'Mary Celeste

In [ ]:
for i in rem_keys:
  _ = c1_pages_db.pop(i,None)

In [ ]:
"Mary Celeste" in cp_pages_db.keys()

True

In [ ]:
rem_keys = []
for key, value in cp_pages_db.items():
  if value['origin_title'] in erase_origin:
    origin = value['origin_title']
    if key == origin[9:]:
      print("spared:"+key)
    else:
      rem_keys.append(keys)
print(len(rem_keys))
for i in rem_keys:
  _ = cp_pages_db.pop(i,None)

NameError: name 'cp_pages_db' is not defined

In [ ]:
_ = cp_pages_db.pop('Moran family',None)

In [ ]:
error guard
wikipedia_queue = deque(to_add_page)
MAX_DEPTH = 1
run_crawler(limit=10000,force_visit=True) # Adjust limit as needed
save_progress() # Save results

Crawling: Black Dahlia
Crawling: Murder of Tupac Shakur
Crawling: Mary Celeste
Crawling: Disappearance of Don Banfield
Crawling: Jack the Ripper
Crawling: Malaysia Airlines Flight 370
Finished while loop
--- Progress Saved Successfully ---
Crawler stopped. Progress has been secured.
--- Progress Saved Successfully ---


In [ ]:
for p in to_add_page:
  content = pages_db.get(p[0],None)
  if content:
    cp_pages_db[p[0]] = content
  else:
    print("Not found "+p[0])

Not found Black_Dahlia
Not found Murder_of_Tupac_Shakur


In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_1.json", "w", encoding="utf-8") as f:
    json.dump(cp_pages_db, f, ensure_ascii=False, indent=4)

### Sections Cleaning

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_1.json", "r", encoding="utf-8") as f:
    c1_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
len(c1_pages_db)

5076

In [ ]:
erase_origin = ["Category:Castaways","Category:Highway of Tears","Category:Jack the Ripper","Category:Kidnapped Palestinian people",
                "Category:Lost and extinct musical instruments","Category:Melbourne gangland killings","Category:Moran family",
                "Category:POW/MIA advocacy","Category:Search for Malaysia Airlines Flight 370","Category:Vietnam War POW/MIA issues",
                "Category:Vietnam War POW/MIA issues","Category:Victims of the Melbourne gangland killings","Category:Works about missing people"]

In [ ]:
rm_keys = []
for keys, value in c1_pages_db.items():
  if value['origin_title'] in erase_origin:
    print(keys)
    rm_keys.append(keys)

Asor
Chekker
Chrotta
Giga (instrument)
Gingras (instrument)
Gittith
Gue
Nevel (instrument)
Sambuca (instrument)
Suka (string instrument)
Symphonia
Poirot Investigates
The Labours of Hercules
Complement of HMS Bounty
Fat Tony & Co.
Keith Faure
Zarah Garde-Wilson
Mick Gatto
Evangelos Goussis
Murders of Terrence and Christine Hodson
Informer 3838
Tony Mokbel
Underbelly series 1
Carl Williams (criminal)
Highway of Tears
British Columbia Highway 16
Finding Dawn
Bobby Jack Fowler
Highway of Tears (film)
Edward Dennis Isaac
Cody Legebokoff
Bluefin-21
Joint Agency Coordination Centre
Ocean Infinity
West Ridge (ship)
Killing of Rami Ayyad
Kidnapping and murder of Mohammed Abu Khdeir
National Alliance of Families
National League of POW/MIA Families
POW bracelet
Rolling Thunder (organization)
The Saluting Marine
Ted Sampley
Vietnam War POW/MIA issue
Ark Project
Australians missing in action in the Vietnam War
Baron 52
The Code of Conduct and the Vietnam Prisoners of War
Defense Prisoner of War/Mi

In [ ]:
rm_keys

['Asor',
 'Chekker',
 'Chrotta',
 'Giga (instrument)',
 'Gingras (instrument)',
 'Gittith',
 'Gue',
 'Nevel (instrument)',
 'Sambuca (instrument)',
 'Suka (string instrument)',
 'Symphonia',
 'Poirot Investigates',
 'The Labours of Hercules',
 'Complement of HMS Bounty',
 'Fat Tony & Co.',
 'Keith Faure',
 'Zarah Garde-Wilson',
 'Mick Gatto',
 'Evangelos Goussis',
 'Murders of Terrence and Christine Hodson',
 'Informer 3838',
 'Tony Mokbel',
 'Underbelly series 1',
 'Carl Williams (criminal)',
 'Highway of Tears',
 'British Columbia Highway 16',
 'Finding Dawn',
 'Bobby Jack Fowler',
 'Highway of Tears (film)',
 'Edward Dennis Isaac',
 'Cody Legebokoff',
 'Bluefin-21',
 'Joint Agency Coordination Centre',
 'Ocean Infinity',
 'West Ridge (ship)',
 'Killing of Rami Ayyad',
 'Kidnapping and murder of Mohammed Abu Khdeir',
 'National Alliance of Families',
 'National League of POW/MIA Families',
 'POW bracelet',
 'Rolling Thunder (organization)',
 'The Saluting Marine',
 'Ted Sampley',
 'V

In [ ]:
for i in rm_keys:
  a = c1_pages_db.pop(i,None)
  if not a:
    print("not remove "+i)

In [ ]:
len(c1_pages_db)

5076

In [ ]:
all_outer_section = set()

def print_outer_section(sections):
  for s in sections:
    #print(prev + s["title"])
    all_outer_section.add(s["title"])

for keys, value in c1_pages_db.items():
  sect_tree = value["sections_tree"]
  #print_section_tree(sect_tree,"")
  print_outer_section(sect_tree)

In [ ]:
all_outer_section = list(all_outer_section)
all_outer_section.sort()
for i in all_outer_section:
  print(i)

"Bodyhunters" documentary
"Brides on Tour" Project
"Fifty Mission Cap"
"French connection"
"Fritz" Johnson
"Hereditary prince" theory
"Historic Truth" version of events
"Rubbish" of Vincent
"Ruislip's Murder Mile"
"Sacrifice" of ritual artwork
"Seacroft"
"Stand your ground" laws
"Thirty-Seven Days of Peril"
"Toxic Truck"
"Yo me niego a rendirme"
#AmINext campaign
#BringBackOurGirls movement and protests
'Bloody April'
'Take It Down' tool
(October 1985) Broadwater Farm riot
10 August 1975
12 February 2002 terror alert
13 December 2008 incident
16 May 2009 incident
161 (Special Duties) Squadron
1776–1783
1831 murder
1865 fire at the Smithsonian
1882 voyage
18F
1900 crew disappearance
1908 Olympic results
1912 Arctic expedition
1917–18
1919 World Series
1921 Travers Stakes
1926 Ford Reliability NationalTour
1927 Circumnavigation of Australia
1928 Trans-Pacific flight
1928 Trans-Tasman flight
1929 "Coffee Royal" tragedy
1930s and beyond
1936 Olympic results
1938 mausoleum looting incident


In [ ]:
error guard
with open("section_name.txt", "w", encoding="utf-8") as f:
    for title in all_outer_section:
        f.write(title + "\n")

In [ ]:
discardable_sections = [
    "Accolades",    "Acknowledgements",    "Additional links",    "Album",    "Albums",    "Appendices",    "Archival collections",    "Archives",    "Archives At",    "Articles",    "Articles or Books on Turner",    "Articles written",    "Autobiography: The Contras' Valley Forge",
    "Award",    "Award Citations",    "Awards",    "Awards and Decorations",    "Awards and accolades",    "Awards and citations",    "Awards and commendation",    "Awards and decorations",    "Awards and honors",    "Awards and honours",    "Awards and memory",    "Awards and nominations",    "Awards and recognition",    "Awards and showings",    "Awards and tributes",    "Awards, decorations, and honors",
    "BAA/NBA career statistics",    "Bibliographies",    "Bibliography",    "Biographical works",    "Book",    "Book publication",    "Books",    "Books about Abani Chakravarty",    "Books about Edwards",    "Books and films about Litvinenko",    "Books and films about the Oakes case",    "Books and movie",    "Books and movies",    "Books and publications",    "Books by Earhart",
    "Books by or about Murray O'Hair",    "Books, documentaries and media",    "Career statistics",    "Championships",    "Championships and accomplishments",    "Cinematic adaptations",    "Citation",    "Citations",    "Citations and references",    "Citations of honors",    "Cited sources",    "Cited works and further reading",
    "Collections",    "Collections and exhibitions",    "Compilation contributions",    "Compilations",    "Compositions",    "Contents",    "Creative works",    "Cultural depictions",    "Cultural references",    "Databases",    "Dedicated taxa",
    "Depictions in art and literature",    "Depictions in fiction",    "Depictions in film",    "Depictions in music and popular culture",    "Depictions in popular culture",    "Discography",    "Discography and writing credits",    "Documentaries",    "Documentary",    "Documentary film",    "Documentary films",    "EP",    "Electoral history",
    "Electoral record",    "Engravings from John Pine's 1739 publication",    "Eponym",    "Eponyms",    "Eponymy",    "Exhibition history",    "Exhibitions",    "Exhibitions, reception and impact",    "Explanatory notes",    "External",    "External Links",    "External link",
    "External links",    "External links and further reading",    "External links and sources",    "External websites",    "Figures",    "Film adaptation",    "Film adaptations",    "Film adaptations and documentaries",    "Film adaptations based upon the events",
    "Filmography",    "Filmography (selection)",    "Films",    "Films about Calvi's death",    "Films about the Johann Orth mystery",    "Films and documentaries",    "Films and music",    "Footnote",    "Footnotes",    "Footnotes and references",    "Foreign honours",
    "Further reading",    "Further references",    "Gallery",    "Genealogical table",    "General and cited references",    "General and cited sources",    "General references",    "General sources",    "George Cross award",    "George Cross citation",    "Heraldry",    "His books",    "Honours",    "Honours and arms",    "Honours and awards",    "Honours and family",
    "Honours and legacy",    "Honours and memorials",    "Honours and recognition",    "Honours and tributes",    "Image gallery",    "In art",    "In art and media",    "In books",    "In fiction",    "In fiction and popular culture",    "In film",    "In literature",    "In literature and the arts",    "In literature and theater",
    "In literature, media and popular culture",    "In media",    "In media and culture",    "In other media and popular culture",    "In popular culture",    "In popular culture and mass media",    "In popular culture and media",    "In popular media",    "Individual publications",    "Informational notes",    "Inline citations",    "Institutions with his works",    "Library",    "List of characters",
    #"List of deceased players",
    "List of tours",    "List of victories",    "List of works",    "Lists of Wikipedia articles on notable destroyed artworks",    "Literary activity",    "Literary archives",
    "Literary career",    "Literary contributions",    "Literary depictions",    "Literary relevance",    "Literary works",    "Literary writer",    "Literature",    "Literature and other cultural influences",
    "Managerial statistics",    "Medal of Honor citation",    "Memorial events",    "Memorial fund",    "Memorial seat",    "Memorial services",    "Memorial stones in Gallehus",    "Memorial surfing invitational",    "Memorials",
    "Memorials and legacy",    "Memorials and other legacies",    "Mixed martial arts record",    "Monuments and memorials",    "Monument to Laporte",    "Monument to Morgan",    "Movies with DeLay",    "Multilingual editions",    "Musical pieces",    "Namesake",    "Namesakes",    "Namesake honors",
    "Navy Cross citation",    "Newspaper references",    "Note",    "Notes",    "Notes and citations",    "Notes and references",    "Novels",    "Other reading",    "Other references",    "Other websites",    "Other works",    "Paintings",    "Patents",    "Patents and cameras",
    "Personal bests",    "Photographs",    "Plays",    "Plot summary",    "Podcasts",    "Podcasts and documentaries",    "Pop culture",    "Popular culture",    "Popular songs",    "Portrayal in film",    "Portrayal in media",    "Portrayal in movies",    "Portrayal in popular media",    "Portrayals",    "Portrayals and cultural references",
    "Portrayals and references in media",    "Portrayals in fiction",    "Portrayals in film",    "Postage stamp",    "Postscript",    "Posthumous awards and honours",    "Posthumous honours",    "Posthumous publications",    "Posthumous releases",
    "Precursor publications",    "Primary sources",    "Private art collections",    "Professional boxing record",    "Public collections (selection)",    "Publications",    "Publications by Berezovsky",    "Published works",    "Published works by Peter V. Verigin",    "Publishing",    "Quotations",    "Quote",    "Quotes",    "Radio career",
    "Radio credits",    "Radio documentaries",    "Radio documentary",    "Recordings",    "References",    "References and further reading",    "References and notes",    "References in Abrahamic religions",    "References in books and plays",    "References in media",    "References in pop culture",    "References in popular culture",    "Related books",    "Related reading",
    "Related works",    "Replica",    "Replica vessels",    "Replicas",    "Replicas and imitations",    "Representation in art and media",    "Representation in film and literature",    "Representation in other media",    "Representation in popular culture",    "Representations in contemporary art",    "Representations in culture and media",    "RSPCA Purple Cross Award",
    "See Also",    "See also",    "Selected articles in collective works",    "Selected bibliography",    "Selected filmography",    "Selected groups",    "Selected media",    "Selected publications",    "Selected videography",    "Selected works",    "Selective list of works",
    "Singles",    "Sister cities",    "Songs about Yokota",    "Sources",    "Sources and bibliograpahy",    "Sources and bibliography",    "Sources and notes",
    "Sources cited",    "Succession table",    "Table of contents",    "Televised documentaries and dramas",    "Television",    "Television career",    "Television documentaries",    "Television documentary",    "Television movies",    "Tribute",    "Tribute website",    "Tributes",    "Tributes and political reactions",    "Tributes and public reaction",    "Tributes to Ananda Mahidol",
    "Tributes to Marin Preda",    "Translation notes",    "Translations",    "Translations of Preda's work",    "Unreleased work",    "Verse",    "Verses",    "Victoria Cross citation",    "Videography",    "Wallpapers",    "Website on Abani Chakravarty",    "Works",
    "Works about Berezovsky",    "Works by Amundsen",    "Works cited",    "Works consulted",    "Works related to García Lorca",    "Works, editions and recordings",    "World Championship results",    "World Cup results",    "Writing",    "Writing and translation",
    "Writings",    "Writings and awards",    "Written works involving Paul Crampel"
]

In [ ]:
discardable_sections = [s.lower() for s in discardable_sections]

In [ ]:
for keys, value in c1_pages_db.items():
  sect_tree = value["sections_tree"]
  value["sections_tree"] = [d for d in sect_tree if d.get('title').lower() not in discardable_sections]

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_2.json", "w", encoding="utf-8") as f:
    json.dump(c1_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
error guard
all_sections = set()
def print_section_tree(sections, prev=""):
  for s in sections:
    #print(prev + s["title"])
    all_sections.add(prev + s["title"])
    for c in s["content"]:
      if type(c) is list:
        print_section_tree(c,prev=f"{prev}{s["title"]} > ")

print(len(all_sections))
all_sections = list(all_sections)
all_sections.sort()
for i in all_sections[:5000]:
  print(i)

### Further Page Cleaning

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_2.json", "r", encoding="utf-8") as f:
    c2_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
all_title = list(c2_pages_db.keys())
with open("page_title.txt", "w", encoding="utf-8") as f:
    for title in all_title:
        f.write(title + "\n")

In [ ]:
page_origin = set()
for key, value in c2_pages_db.items():
  page_origin.add(value['origin_title'])
page_origin = list(page_origin)
page_origin.sort()
for i in page_origin:
  print(i)

Category:1490s missing person cases
Category:1500s missing person cases
Category:1570s missing person cases
Category:1580s missing person cases
Category:1590s missing person cases
Category:1620s missing person cases
Category:1670s missing person cases
Category:1700s missing person cases
Category:1730s missing person cases
Category:1790s missing person cases
Category:17th-century missing person cases
Category:1800s missing person cases
Category:1810s missing person cases
Category:1820s missing person cases
Category:1830s missing person cases
Category:1840s missing person cases
Category:1850s missing person cases
Category:1860s missing person cases
Category:1870s missing person cases
Category:1880s missing person cases
Category:1890s missing person cases
Category:1900s missing person cases
Category:1910s missing person cases
Category:1920s missing person cases
Category:1930s missing person cases
Category:1940s missing person cases
Category:1950s missing person cases
Category:1960s missin

In [ ]:
discarded_pages = [
    "Cold case",
    "Missing person",
    "Office on Missing Persons",
    "Anthropological criminology",
    "Presumption of death",
    "Bermuda",
    "Sargasso Sea",
    "Template:Bermuda Triangle",
    "Missing children",
    "Unreported missing",
    "Missing white woman syndrome",
    "National Missing Children's Day",
    "Latter Day Saint movement and engraved metal plates",
    "Terrorism in Canada",
    "Miscarriage of justice",
    "Draft:Santa Fe County Jane Doe",
    "Michael Collins (Irish leader)",
    "Zachary Taylor",
    "Crime clearance rate",
    "Crime statistics",
    "Cybercrime",
    "Samuel Liddell MacGregor Mathers",
    "Emil Hácha",
    "Crime mapping",
    "Crime science",
    "National Incident-Based Reporting System",
    "Dark figure of crime",
    "Bureau of Justice Statistics",
    "American Society of Criminology",
    "Quantitative methods in criminology",
    "Computational criminology",
    "Victimisation",
    "Cultural criminology",
    "Unsolved murder",
    "Bibliography of sociology",
    "Criminology",
    "Evidence-based policing",
    "Women's fear of crime",
    "Crime harm index",
    "Predictive policing",
    "Unsolved crime",
    "Template:Criminology",
    "Shawn Hornbeck Foundation",
    "Asian Federation Against Involuntary Disappearances",
    "The Emergency (India)",
    "National Commission for the Missing (Syria)",
    "MIA Hunters",
    "Hainan Island incident",
    "Missing in action",
    "Madres buscadoras",
    "Abuja Area Mama",
    "Cebu City",
    "Manchester, Calgary",
    "Prostitution in Rwanda",
    "Rainbow Honor Walk",
    "Adventures with Purpose",
    "Aguilas del Desierto",
    "Black and Missing Foundation",
    "Defense POW/MIA Accounting Agency",
    "The Doe Network",
    "FBI Laboratory",
    "International Centre for Missing & Exploited Children",
    "Laura Recovery Center",
    "Mattie's Call",
    "National Center for Missing & Exploited Children",
    "National Child Victim Identification Program",
    "National Crime Information Center",
    "National Missing and Unidentified Persons System",
    "Office of Children's Issues",
    "Office to Monitor and Combat Trafficking in Persons",
    "Outpost for Hope",
    "Polly Klaas Foundation",
    "Sahnish Scouts",
    "Silver Alert",
    "Texas EquuSearch",
    "Trans Doe Task Force",
    "University of North Texas Center for Human Identification",
    "DNA Doe Project",
    "Reported Missing (TV series)",
    "Vanished (British TV series)",
    "Sensing Murder",
    "Fear of crime",
    "Jōhatsu",
    "Mayer Daak",
    "Voice for Baloch Missing Persons",
    "FUNDENL",
    "Grandmothers of Plaza de Mayo",
    "HIJOS",
    "Independent Commission for the Location of Victims' Remains",
    "Henry VII Chapel",
    "Ludlow Castle",
    "St John's Abbey, Colchester",
    "St Stephen's Chapel",
    "St George's Chapel, Windsor Castle",
    "Titulus Regius",
    "History of the Indian Tribes of North America",
    "Harley Street",
    "Zoroaster",
    "Judas Iscariot",
    "Peter III of Russia",
    "Haile Selassie",
    "Venustiano Carranza",
    "Liaquat Ali Khan",
    "Michael Collins (Irish leader)",
    "Pat Garrett",
    "Adolph Dubs",
    "Charles XII of Sweden",
    "Agnès Sorel",
    "Zannanza",
    "Prostitution in Rwanda"
]

In [ ]:
for p in discarded_pages:
  a = c2_pages_db.pop(p,None)
  if not a:
    print("not found "+p)

not found Michael Collins (Irish leader)
not found Prostitution in Rwanda


In [ ]:
error guard
#manual
a = c2_pages_db.pop("Prostitution in Rwanda",None)
a

In [ ]:
c2_pages_db["Jack the Ripper"]

{'origin_title': 'Unidentified British serial killers',
 'url': 'https://en.wikipedia.org/wiki/Jack_the_Ripper',
 'last_update_text': 'This page was last edited on 10 May 2026, at 03:41(UTC).',
 'summary': 'Jack the Ripper was an unidentified serial killer active in and around the impoverished Whitechapel district of London, England, in 1888. In both criminal case files and the contemporaneous journalistic accounts, the killer was also called the Whitechapel Murderer and Leather Apron.\nAttacks ascribed to Jack the Ripper typically involved women working as prostitutes who lived in the slums of the East End of London. Their throats were cut prior to abdominal mutilations. The removal of internal organs from at least three of the victims led to speculation that the killer had some anatomical or surgical knowledge. Rumours that the murders were connected intensified in September and October 1888, and numerous letters were received by media outlets and Scotland Yard from people purporting

In [ ]:
erase_origin = ["Category:Malaysia Airlines Flight 370"]

In [ ]:
rem_keys = []
for key, value in c2_pages_db.items():
  if value['origin_title']=="Category:Malaysia Airlines Flight 370":
    origin = value['origin_title']
    if key == origin[9:]:
      print("spared:"+key)
    else:
      rem_keys.append(key)
print(len(rem_keys))


0


In [ ]:
for i in rem_keys:
  _ = c2_pages_db.pop(i,None)

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_3.json", "w", encoding="utf-8") as f:
    json.dump(c2_pages_db, f, ensure_ascii=False, indent=4)

### Categories

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_3.json", "r", encoding="utf-8") as f:
    c3_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
all_title = list(c3_pages_db.keys())
all_title.sort()
with open("page_title.txt", "w", encoding="utf-8") as f:
    for title in all_title:
        f.write(title + "\n")

In [ ]:
murder_keywords = [
    "murder", "killing", "massacre", "assassination", "homicide",
    "shooting", "bombing", "arson", "strangler", "killer", "ripper", "poisoning",
]

missing_keywords = [
    "disappearance", "missing", "kidnapping", "abduction",
    #"lost", "theft", "heist", "robbery", "escape", "stolen", "disappeared"
]

unexplained_death_keywords = [
    "death"
]

In [ ]:
for key, value in c3_pages_db.items():
  if any(word in key.lower() for word in murder_keywords):
    value["SE_Category"] = "murder"
  elif any(word in key.lower() for word in missing_keywords):
    value["SE_Category"] = "missing"
  elif any(word in key.lower() for word in missing_keywords):
    value["SE_Category"] = "unexplained death"
  else:
    value["SE_Category"] = "-"

In [ ]:
blank = 0
for key, value in c3_pages_db.items():
  if value["SE_Category"]=="-":
    blank+=1
print(blank)

3511


In [ ]:
error guard
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4a.json", "w", encoding="utf-8") as f:
    json.dump(c3_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4a.json", "r", encoding="utf-8") as f:
    c4a_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
all_origin = set()
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    all_origin.add(value['origin_title'])
all_origin = list(all_origin)
all_origin.sort()
with open("page_origin.txt", "w", encoding="utf-8") as f:
    for title in all_origin:
        f.write(title + "\n")

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if 'murder' in value['origin_title'].lower():
      value["SE_Category"] = "murder"
    elif ('missing' in value['origin_title'].lower() or 'disappear' in value['origin_title'].lower()) and 'people' in value['origin_title'].lower():
      value["SE_Category"] = "missing"
    elif 'death' in value['origin_title'].lower():
      value["SE_Category"] = "unexplained death"

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if 'missing' in value['origin_title'].lower() and 'person' in value['origin_title'].lower():
      value["SE_Category"] = "missing"

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if 'enforced disappearances' in value['origin_title'].lower():
      value["SE_Category"] = "missing"

In [ ]:
origin_missing = ['Category:Aerial disappearances',
 'Category:Aerial disappearances of military personnel in action','Category:Disappeared journalists','Category:Lost explorers', 'Category:Mass disappearances',
 'Category:Missing American children', 'Category:Missing German children', 'Category:Missing air passengers',
 'Category:Missing aircraft', 'Category:Missing aviators',
 'Category:Missing children','Category:Lost sailing vessels','Category:Missing in action of World War I',
 'Category:Missing in action of World War II','Category:Missing ships',
 'Category:Missing ships of Australia', 'Category:Missing submarines',
 'Category:Missing submarines of World War II', 'Category:Missing trains','Category:People declared dead in absentia',
 'Category:People lost at sea','Category:Unexplained_disappearances','Missing aircraft',
 'Unexplained disappearances','Category:Unexplained_disappearances','List of missing ships',"Category:Missing scientists investigation"]

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if value['origin_title'] in origin_missing:
      value["SE_Category"] = "missing"

In [ ]:
origin_wanted = ['Category:Missing fugitives', 'Category:Missing gangsters','Category:Terrorist incidents by unknown perpetrators',
 'Category:Unidentified American criminals', 'Category:Unidentified American rapists', 'Category:Unidentified American serial killers',
 'Category:Unidentified British criminals', 'Category:Unidentified British rapists', 'Category:Unidentified Canadian criminals',
 'Category:Unidentified criminals', 'Category:Unidentified rapists', 'Category:Unidentified serial killers','List_of_fugitives_from_justice_who_disappeared']

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if value['origin_title'] in origin_wanted:
      value["SE_Category"] = "wanted"

In [ ]:
origin_treasure = ['Category:Lost buildings and structures', 'Category:Lost objects', 'Category:Lost works of art']
origin_legend = ['Category:Bermuda Triangle']

In [ ]:
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    if value['origin_title'] in origin_treasure:
      value["SE_Category"] = "treasure"
    elif value['origin_title'] in origin_legend:
      value["SE_Category"] = "legend"

In [ ]:
a = c4a_pages_db.pop("National Biodefense Analysis and Countermeasures Center",None)
a

{'origin_title': 'Category:2001 anthrax attacks',
 'url': 'https://en.wikipedia.org/wiki/National_Biodefense_Analysis_and_Countermeasures_Center',
 'last_update_text': 'This page was last edited on 18 May 2026, at 12:09(UTC).',
 'summary': 'The National Biodefense Analysis and Countermeasures Center (NBACC) is a government biodefense research laboratory created by the U.S. Department of Homeland Security (DHS) and located at the sprawling biodefense campus at Fort Detrick in Frederick, MD, USA. The NBACC (pronounced EN-back) is the principal U.S. biodefense research institution engaged in laboratory-based threat assessment and bioforensics.  NBACC is an important part of the National Interagency Biodefense Campus (NIBC) also located at Fort Detrick for the US Army, National Institutes of Health and the US Department of Agriculture.',
 'infobox': {'Formed': '2002; 24\xa0years ago(2002)',
  'Jurisdiction': 'Federal government of the United States',
  'Headquarters': 'Fort Detrick,Frederi

In [ ]:
error guard
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4b.json", "w", encoding="utf-8") as f:
    json.dump(c4a_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
#============================

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4b.json", "r", encoding="utf-8") as f:
    c4b_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
len(c4b_pages_db)

4939

In [ ]:
rm_keys = []
for keys, value in c4b_pages_db.items():
  if value['origin_title']=="Category:2001 anthrax attacks":#'Category:Richard of Shrewsbury, Duke of York':
    rm_keys.append(keys)
for i in rm_keys:
  _ = c4b_pages_db.pop(i,None)

In [ ]:
#Checking
#all_title = set()
for key, value in c4b_pages_db.items():
  if value["SE_Category"] == "-":
    if any(('death' in cat.lower() or 'murder' in cat.lower() or 'killing' in cat.lower()) for cat in value['categories']):
      print(f"{key} < {value['origin_title']}")
      value["SE_Category"] = 'murder'
    #value['categories'] any(word in key.lower() for word in murder_keywords):
    #all_title.add(key)
#all_title = list(all_title)
#all_title.sort()
#all_title

Elfrieda Knaak < Category:Unsolved crimes
Harry Roy Veevers < Category:Unsolved crimes
Cairo fire < Category:Unsolved arsons
1978 Holiday Inn fire < Category:Unsolved arsons
Myojo 56 building fire < Category:Unsolved arsons
2020 New York City Subway fire < Category:Unsolved arsons
Stouffer's Inn fire < Category:Unsolved arsons
The Gentleman of Heligoland < Category:Unsolved crimes in Europe
Operation Identify Me < Category:Unsolved crimes in Europe
Mr Cruel < Category:Unsolved crimes in Australia
Hummingbird heist < Category:Unsolved crimes in Belize
El Psicópata < Category:Unsolved crimes in Costa Rica
Francis Imbard affair < Category:Unsolved crimes in France
Prithipal Singh < Category:Unsolved crimes in India
Death of Benny Whitehouse < Category:Unsolved crimes in Ireland
Stolero-Yachman-Chachkes affair < Category:Unsolved crimes in Israel
San Juan del Sur Psychopath < Category:Unsolved crimes in Nicaragua
Ekiti prison break < Category:Unsolved crimes in Nigeria
Haji Sulong < Catego

In [ ]:
#Checking
#all_title = set()
for key, value in c4b_pages_db.items():
  if value["SE_Category"] == "-":
    if any(('theft' in cat.lower() or 'lost object' in cat.lower() or 'robbery' in cat.lower()) for cat in value['categories']):
      print(f"{key} < {value['origin_title']}")
      value["SE_Category"] = 'treasure'
    #value['categories'] any(word in key.lower() for word in murder_keywords):
    #all_title.add(key)
#all_title = list(all_title)
#all_title.sort()
#all_title

Corrida (horse) < Category:Unsolved crimes in Europe
1983 theft of the Jules Rimet Trophy < Category:Unsolved crimes in Brazil
1972 Montreal Museum of Fine Arts robbery < Category:Unsolved crimes in Canada
2011 Montreal Museum of Fine Arts theft < Category:Unsolved crimes in Canada
Irish Crown Jewels < Category:Unsolved crimes in Ireland
Biblioteca della Comunità Israelitica < Category:Unsolved crimes in Italy
2025 Drents Museum heist < Category:Unsolved crimes in the Netherlands
1966 theft of the Jules Rimet Trophy < Category:Unsolved crimes in the United Kingdom
2022 Brink's theft < Category:Unsolved crimes in the United States
Easter Sunday heist < Category:Unsolved crimes in the United States
Isabella Stewart Gardner Museum theft < Category:Unsolved crimes in the United States


In [ ]:
c4b_pages_db['Eastcastle Street robbery']["SE_Category"] = 'treasure'
c4b_pages_db['300 million yen robbery']["SE_Category"] = 'treasure'
c4b_pages_db['1987 Opera House heist']["SE_Category"] = 'treasure'
c4b_pages_db['1977 Krugersdorp bank robbery']["SE_Category"] = 'treasure'
c4b_pages_db['1999 Loomis truck robbery']["SE_Category"] = 'treasure'

In [ ]:
c4b_pages_db['Gajraula Nuns rape case']["SE_Category"] = 'wanted'

In [ ]:
c4b_pages_db['Woman-Ochre']["SE_Category"] = 'treasure'

In [ ]:
c4b_pages_db['Christiana Mall']["SE_Category"] = 'murder'

In [ ]:
c4b_pages_db['Christiana Mall']["SE_Category"] = 'missing'

In [ ]:
error guard
if c4b_pages_db.pop('Hans van de Kimmenade',None):
  print("deleted")

deleted


In [ ]:
blank = 0
for key, value in c4b_pages_db.items():
  if value["SE_Category"]=="-":
    blank+=1
print(blank)

0


In [ ]:
list_se_cat = []
for key, value in c4b_pages_db.items():
  list_se_cat.append(value["SE_Category"])
distribution = Counter(list_se_cat)
print(distribution)

Counter({'missing': 2706, 'murder': 1719, 'unexplained death': 221, 'wanted': 187, 'treasure': 87, 'legend': 8})


In [ ]:
c4b_pages_db.pop("Aguanga, California",None)

{'origin_title': 'Category:Unsolved mass murders in California',
 'url': 'https://en.wikipedia.org/wiki/Aguanga,_California',
 'last_update_text': 'This page was last edited on 16 May 2026, at 23:29(UTC).',
 'summary': 'Aguanga (; Luiseño: Awáanga, meaning "dog place") is a census-designated place located within the Inland Empire in Riverside County, California.  It is located about 18 miles (29 km) east of Temecula and 22 miles (35 km) south-southeast of Hemet. Aguanga lies at an elevation of 1955 feet (596 m). As of the 2020 census, it had a population of 989.',
 'infobox': {'Country': 'United States',
  'State': 'California',
  'County': 'Riverside',
  '•\xa0Total': '989',
  '•\xa0Land': '13.60\xa0sq\xa0mi (35.22\xa0km2)',
  '•\xa0Water': '0\xa0sq\xa0mi (0.00\xa0km2) \xa00%',
  'Elevation[2]': '1,955\xa0ft (596\xa0m)',
  '•\xa0Density': '72.7/sq\xa0mi (28.08/km2)',
  'Time zone': 'UTC-8(Pacific (PST))',
  '•\xa0Summer (DST)': 'UTC-7(PDT)',
  'ZIP codes': '92536',
  'Area code': '951

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4c.json", "w", encoding="utf-8") as f:
    json.dump(c4b_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
#Checking
all_title = set()
for key, value in c4b_pages_db.items():
  if value["SE_Category"] == "-":
    all_title.add(key)
all_title = list(all_title)
all_title.sort()
all_title

[]

In [ ]:
#Checking
for key, value in c4a_pages_db.items():
    if value['origin_title']=='Category:Unsolved crimes in England' and value["SE_Category"] == "-":
      print(key)

Gatwick Airport drone incident


In [ ]:
#Checking
all_origin = set()
for key, value in c4a_pages_db.items():
  if value["SE_Category"] == "-":
    all_origin.add(value['origin_title'])
all_origin = list(all_origin)
all_origin.sort()
all_origin

['Category:2001 anthrax attacks',
 'Category:Richard of Shrewsbury, Duke of York',
 'Category:Unsolved airliner bombings',
 'Category:Unsolved airliner bombings in the United States',
 'Category:Unsolved arsons',
 'Category:Unsolved crimes',
 'Category:Unsolved crimes in Australia',
 'Category:Unsolved crimes in Belize',
 'Category:Unsolved crimes in Brazil',
 'Category:Unsolved crimes in Canada',
 'Category:Unsolved crimes in China',
 'Category:Unsolved crimes in Costa Rica',
 'Category:Unsolved crimes in England',
 'Category:Unsolved crimes in Europe',
 'Category:Unsolved crimes in France',
 'Category:Unsolved crimes in Germany',
 'Category:Unsolved crimes in India',
 'Category:Unsolved crimes in Ireland',
 'Category:Unsolved crimes in Israel',
 'Category:Unsolved crimes in Italy',
 'Category:Unsolved crimes in Japan',
 'Category:Unsolved crimes in Nicaragua',
 'Category:Unsolved crimes in Nigeria',
 'Category:Unsolved crimes in South Africa',
 'Category:Unsolved crimes in Thailand',

### Location

In [ ]:
cc = coco.CountryConverter()

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4c.json", "r", encoding="utf-8") as f:
    c4c_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
for keys, value in c4c_pages_db.items():
  infobox = value['infobox']
  infobox_country = infobox.get('Country',None)
  infobox_location = infobox.get('Location',None)
  infobox_place = infobox.get('Place',None)
  se_location = "-"
  full_location = "-"
  if infobox_country:
    country = cc.convert(infobox_country,to = 'name')
    if country != 'not found':
      se_location = cc.convert(infobox_country,to = 'name')
      if infobox_location:
        full_location = infobox_location
        if cc.convert(infobox_location[-1])=='not found':
          full_location += f", {se_location}"
  elif infobox_location:
    full_location = infobox_location
    parts = [p.strip() for p in full_location.split(',')]
    country = cc.convert(parts[-1],to = 'name')
    if country != 'not found':
      se_location = country
  elif infobox_place:
    full_location = infobox_place
    parts = [p.strip() for p in full_location.split(',')]
    country = cc.convert(parts[-1],to = 'name')
    if country != 'not found':
      se_location = country
  value['SE_Location'] = se_location
  value['Full Location'] = full_location

In [ ]:
for keys, value in c4c_pages_db.items():
  if value['SE_Location']=="-":
    if value['origin_title'].startswith('Category:Missing person cases in '):
      country = cc.convert(value['origin_title'][33:],to = 'name')
      if country != 'not found':
        value['SE_Location'] = country
        value['Full Location'] = country
    elif value['origin_title'].startswith('Category:Enforced disappearances in '):
      country = cc.convert(value['origin_title'][36:],to = 'name')
      if country != 'not found':
        value['SE_Location'] = country
        value['Full Location'] = country
    elif value['origin_title'].startswith('Category:Unsolved crimes in '):
      country = cc.convert(value['origin_title'][28:],to = 'name')
      if country != 'not found':
        value['SE_Location'] = country
        value['Full Location'] = country

In [ ]:
for keys, value in c4c_pages_db.items():
  if value['SE_Location']=="-":
    if value['origin_title'].startswith('Category:Unsolved murders in '):
      country = cc.convert(value['origin_title'][29:],to = 'name')
      if country != 'not found':
        value['SE_Location'] = country
        value['Full Location'] = country

In [ ]:
map_l ={
    "Category:Missing person cases in London":('United Kingdom','London, United Kingdom'),
    "Category:Missing person cases in Scotland":('United Kingdom','United Kingdom'),
    "Category:Missing person cases in Wales":('United Kingdom','United Kingdom'),
    "Category:Unidentified American criminals":("United States","United States"),
    "Category:Unidentified American rapists":("United States","United States"),
    "Category:Unidentified British rapists":('United Kingdom','United Kingdom'),
    "Category:Unidentified British serial killers":('United Kingdom','United Kingdom'),
    "Category:Unidentified Canadian criminals":('Canada','Canada'),
    "Category:Unidentified murder victims in the United Kingdom":('United Kingdom','United Kingdom'),
    "Category:Unidentified murder victims in the United States":("United States","United States"),
    "Category:Unsolved airliner bombings in the United States":("United States","United States"),
    "Category:Unsolved mass murders in California":("United States","United States"),
    "Category:Unsolved mass murders in Georgia (U.S. state)":("United States","United States"),
    "Category:Unsolved mass murders in Northern Ireland":('United Kingdom','United Kingdom'),
    "Category:Unsolved mass murders in the United States":("United States","United States"),
"Category:Unsolved murders in London":('United Kingdom','United Kingdom'),
"Category:Unsolved murders in Northern Ireland":('United Kingdom','United Kingdom'),
"Category:Unsolved murders in Scotland":('United Kingdom','United Kingdom'),
"Category:Unsolved murders in Utah Territory":("United States","United States"),
"Category:Unsolved murders in Wales":('United Kingdom','United Kingdom'),
"Category:Unsolved murders in the Soviet Union":("United States","United States"),
"Category:Missing person cases in Northern Ireland":('United Kingdom','United Kingdom')

}
map1 = {
    'Category:Unidentified American serial killers': 'United States',
'Category:Unidentified British criminals': 'United Kingdom',
    'Category:Missing American children': 'United States',
    'Category:Missing American people': 'United States',
    'Category:Missing Argentine people': 'Argentina',
    'Category:Missing Austrian people': 'Austria',
    'Category:Missing Belgian people': 'Belgium',
    'Category:Missing Brazilian people': 'Brazil',
    'Category:Missing British people': 'United Kingdom',
    'Category:Missing Canadian people': 'Canada',
    'Category:Missing Chinese people': 'China',
    'Category:Missing Colombian people': 'Colombia',
    'Category:Missing English people': 'United Kingdom',
    'Category:Missing Finnish people': 'Finland',
    'Category:Missing French children': 'France',
    'Category:Missing French people': 'France',
    'Category:Missing German children': 'Germany',
    'Category:Missing German people': 'Germany',
    'Category:Missing Indian people': 'India',
    'Category:Missing Indonesian people': 'Indonesia',
    'Category:Missing Iranian people': 'Iran',
    'Category:Missing Irish people': 'Ireland',
    'Category:Missing Israeli people': 'Israel',
    'Category:Missing Italian children': 'Italy',
    'Category:Missing Italian people': 'Italy',
    'Category:Missing Japanese people': 'Japan',
    'Category:Missing Malaysian people': 'Malaysia',
    'Category:Missing New Zealand people': 'New Zealand',
    'Category:Missing Nigerian people': 'Nigeria',
    'Category:Missing Pakistani people': 'Pakistan',
    'Category:Missing Palestinian people': 'Palestine',
    'Category:Missing Russian people': 'Russia',
    'Category:Missing Scottish people': 'United Kingdom',
    'Category:Missing South African people': 'South Africa',
    'Category:Missing South Korean people': 'South Korea',
    'Category:Missing Swedish people': 'Sweden',
    'Category:Missing Swiss people': 'Switzerland',
    'Category:Missing Syrian people': 'Syria',
    'Category:Missing Ugandan people': 'Uganda',
    'Category:Missing Ukrainian people': 'Ukraine'
}

In [ ]:
for keys, value in c4c_pages_db.items():
  if value['SE_Location']!="-":
    continue
  if value['origin_title'] in map_l:
    c2 = map_l[value['origin_title']]
    value['SE_Location'] =c2[0]
    value['Full Location'] =c2[1]
  elif value['origin_title'] in map1:
    c = map1[value['origin_title']]
    value['SE_Location'] =c
    value['Full Location'] =c

In [ ]:
count=0
for keys, value in c4c_pages_db.items():
  if value['SE_Location']=="-":
    #print(keys)
    #break
    count+=1
print(count)

3218


In [ ]:
origins = set()
for keys, value in c4c_pages_db.items():
  if value['SE_Location']=="-":
    origins.add(value['origin_title'])
origins = list(origins)
origins.sort()
for i in origins:
  print(i)

Category:1490s missing person cases
Category:1500s missing person cases
Category:1570s missing person cases
Category:1580s missing person cases
Category:1590s missing person cases
Category:1620s missing person cases
Category:1670s missing person cases
Category:1700s missing person cases
Category:1730s missing person cases
Category:1790s missing person cases
Category:17th-century missing person cases
Category:1800s missing person cases
Category:1810s missing person cases
Category:1820s missing person cases
Category:1830s missing person cases
Category:1840s missing person cases
Category:1850s missing person cases
Category:1860s missing person cases
Category:1870s missing person cases
Category:1880s missing person cases
Category:1890s missing person cases
Category:1900s missing person cases
Category:1910s missing person cases
Category:1920s missing person cases
Category:1930s missing person cases
Category:1940s missing person cases
Category:1950s missing person cases
Category:1960s missin

In [ ]:
error guard
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4d.json", "w", encoding="utf-8") as f:
    json.dump(c4c_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
c4c_pages_db.pop("Operation Identify Me",None)

{'origin_title': 'Category:Unsolved crimes in Europe',
 'url': 'https://en.wikipedia.org/wiki/Operation_Identify_Me',
 'last_update_text': 'This page was last edited on 19 May 2026, at 05:33(UTC).',
 'summary': 'Operation Identify Me was launched on 10 May 2023 by Interpol to solve cold cases across Western Europe to identify 22 unidentified women who were found deceased in the Netherlands, Belgium and Germany between 1976 and 2019. Most of the women were murdered, and none had ever been identified.\nA public appeal was made for information surrounding the unidentified women. Interpol alongside Dutch, German, and Belgian police forces released forensic facial reconstructions as well as other information used in the investigations. It is believed some of the murdered women may be from parts of Eastern Europe.\nThe second phase of the project was launched in October 2024. The 46 newly publicised cases were expanded to France, Italy and Spain.\n\nSince its inception, five women have been 

In [ ]:
page_with_hole = set()

In [ ]:
def clean_sections_recursive(keys, sections, path=""):
    # Iterate backwards so we can delete while looping
    for i in range(len(sections) - 1, -1, -1):
        section = sections[i]
        title = section.get('title', 'Unknown')
        current_path = f"{path} > {title}" if path else title

        content_list = section.get('content', [])

        # 1. Look for a list inside 'content' (these are the subsections)
        for j in range(len(content_list) - 1, -1, -1):
            if isinstance(content_list[j], list):
                # Recurse into the subsections
                clean_sections_recursive(keys, content_list[j], current_path)

                # If the subsections list is now empty, remove it from content
                if len(content_list[j]) == 0:
                    content_list.pop(j)

        # 2. Check if the section is now "Empty"
        # Criteria: content is just [""] or is completely empty []
        is_only_empty_string = len(content_list) == 1 and content_list[0] == ""
        is_completely_empty = len(content_list) == 0

        if is_only_empty_string or is_completely_empty:
            print(f"Deleted empty section: {current_path}")
            found_table = False
            #for table in c4c_pages_db[keys]['table']:
            #  if table["section"]==title:
            #    found_table = True
            #    break
            #if not found_table:
            #  print(f"Deleted {keys} [empty]:{current_path}")
              #page_with_hole.add(keys)
              #sections.pop(i)


In [ ]:
backup = c4c_pages_db.copy()

In [ ]:
for keys, value in c4c_pages_db.items():
  #if not keys in rm_holes:
  #  continue
  section_tree = value["sections_tree"]
  clean_sections_recursive(keys,section_tree)

Deleted empty section: The firefighters
Deleted empty section: Timeline
Deleted empty section: Robberies
Deleted empty section: Victims
Deleted empty section: Motorsports career results > NASCAR > Craftsman Truck Series
Deleted empty section: Supreme Court decision > Composition
Deleted empty section: Known victims > Cases that were included, but later ruled out
Deleted empty section: Known victims > Cases linked to the murders
Deleted empty section: Race summary
Deleted empty section: Crews of Flight 19 and PBM-5 BuNo 59225
Deleted empty section: Vehicle details
Deleted empty section: Notable search efforts
Deleted empty section: Disappearance theories
Deleted empty section: International competitions
Deleted empty section: Windisch victories
Deleted empty section: Tables > Second raiding cruise
Deleted empty section: Tables > First raiding cruise
Deleted empty section: List of Wikipedia articles on notable finds
Deleted empty section: Legal proceedings > Convictions and sentences
Del

In [ ]:
error guard
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4e.json", "w", encoding="utf-8") as f:
    json.dump(c4c_pages_db, f, ensure_ascii=False, indent=4)

### Image, Table, and Cleaning Text

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4e.json", "r", encoding="utf-8") as f:
    c4e_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
WAIT_RANGE = (1,3)

In [ ]:
start_index = 2892

In [ ]:
"images" in c4e_pages_db["Elfrieda Knaak"]

True

In [ ]:
error_page = []

In [ ]:
error_page

['Tracy Allard',
 'Michèle Audette',
 'Tanya Kappo',
 'Lesa Semmler',
 'Dawn Dumont']

In [ ]:
import requests
import json
import re
from bs4 import BeautifulSoup

def get_span_value(cell, attr):
    """Extracts the leading integer from a span attribute, handling malformed HTML."""
    val = cell.get(attr, "1")
    # Match only the digits at the beginning of the string
    match = re.match(r'^(\d+)', str(val).strip())
    return int(match.group(1)) if match else 1

def record_page_tables_as_json(title):
    headers = {'User-Agent': 'MysteryBotName (email@example.com)'}
    url = f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    content = soup.find('div', class_='mw-parser-output')
    if not content: return []

    scraped_tables = []
    current_hierarchy = {2: "Introduction", 3: None, 4: None, 5: None, 6: None}

    for tag in content.find_all(['h2', 'h3', 'h4', 'h5', 'h6', 'table']):
        if tag.name in ['h2', 'h3', 'h4', 'h5', 'h6']:
            level = int(tag.name[1])
            current_hierarchy[level] = tag.get_text(strip=True).replace('[edit]', '')
            for i in range(level + 1, 7): current_hierarchy[i] = None

        elif tag.name == 'table' and 'wikitable' in tag.get('class', []):
            caption_tag = tag.find('caption')
            table_obj = {
                "section_hierarchy": [current_hierarchy[i] for i in range(2, 7) if current_hierarchy[i]],
                "caption": caption_tag.get_text(strip=True) if caption_tag else None,
                "rows": []
            }

            for tr in tag.find_all('tr'):
                row_cells = []
                for cell in tr.find_all(['td', 'th']):
                    row_cells.append({
                        "text": cell.get_text(strip=True),
                        "is_header": cell.name == 'th',
                        "rowspan": get_span_value(cell, 'rowspan'),
                        "colspan": get_span_value(cell, 'colspan')
                    })
                if row_cells:
                    table_obj["rows"].append(row_cells)

            scraped_tables.append(table_obj)

    return scraped_tables

In [ ]:
table = record_page_tables_as_json('Michèle Audette')

In [ ]:
for keys in error_page:
  value = c4e_pages_db[keys]
  table = record_page_tables_as_json(keys)
  imgs, error = get_img(keys)
  if error:
      crawl_errors[keys] = error
  value["table"] = table
  value["images"]= imgs
  count += 1
  if count > batch:
    count = 0
    time.sleep(random.uniform(*WAIT_RANGE))

In [ ]:

for keys, value in tqdm(c4e_pages_db.items()):
    if not "images" in value:
      print(keys)
      break


100%|██████████| 4926/4926 [00:00<00:00, 286860.69it/s]


In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4f.json", "w", encoding="utf-8") as f:
    json.dump(c4e_pages_db, f, ensure_ascii=False, indent=4)

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4f.json", "r", encoding="utf-8") as f:
    c4f_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [ ]:
import re

def clean_all_keys(data):
    """
    Recursively wanders through nested dictionaries and lists to clean strings.
    """
    if isinstance(data, dict):
        return {k: clean_all_keys(v) if not k.lower() in ["website"] else v for k, v in data.items()}
    elif isinstance(data, list):
        return [clean_all_keys(item) for item in data]
    elif isinstance(data, str):
        return transform_text(data)
    else:
        return data

def clean_wikipedia_text(data):
    if isinstance(data, dict):
        # Only clean 'text', 'caption', and 'section_hierarchy'
        # Do NOT clean fields like 'src', 'url', or 'is_header'
        return {
            k: clean_wikipedia_text(v) if k.lower() in ['text', 'caption', 'section_hierarchy'] else v
            for k, v in data.items()
        }
    elif isinstance(data, list):
        return [clean_wikipedia_text(item) for item in data]
    elif isinstance(data, str):
        return transform_text(data)
    return data

def transform_text(text):
    # Remove citations and [edit] tags
    text = re.sub(r'\[\d+\]|\[[a-zA-Z]+\s*\d*\]|\[edit\]', '', text)

    # SAFER Phonetic removal: Look for IPA characters specifically
    # or ensure we aren't stripping a URL (check if it starts with http)
    if not text.startswith('http'):
        text = re.sub(r'\/[^\/]{2,}\/', '', text) # Minimum 2 chars inside slashes

    # Replace weird spaces but KEEP single newlines in tables
    text = text.replace('\xa0', ' ')

    # Collapse multiple horizontal spaces, but preserve single newlines
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

# --- Example for testing ---
wiki_data = {
    "title": "Python (programming language)",
    "content": ["Python was conceived in the late 1980s [1].", "It is used in AI /ˌaɪˈiː/.\xa0[2]"]
}

cleaned_data = clean_all_keys(wiki_data)
print(cleaned_data)

{'title': 'Python (programming language)', 'content': ['Python was conceived in the late 1980s .', 'It is used in AI .']}


In [ ]:
for keys,value in c4f_pages_db.items():
  o_last_update_text = value['last_update_text']
  value['last_update_text'] = transform_text(o_last_update_text)

  o_summary = value['summary']
  value['summary'] = transform_text(o_summary)

  o_infobox = value['infobox']
  value['infobox'] = clean_all_keys(o_infobox)

  o_section_tree = value['sections_tree']
  value['sections_tree'] = clean_all_keys(o_section_tree)

  o_table = value['table']
  value['table'] = clean_wikipedia_text(o_table)

  o_images = value['images']
  value['images'] = clean_wikipedia_text(o_images)

In [ ]:
c4f_pages_db["Christiana Mall"]

{'origin_title': 'Category:Unsolved crimes in the United States',
 'url': 'https://en.wikipedia.org/wiki/Christiana_Mall',
 'last_update_text': 'This page was last edited on 9 May 2026, at 14:36(UTC).',
 'summary': "Christiana Mall is a shopping mall located in Christiana, Delaware between the cities of Newark and Wilmington. The one-level, enclosed super-regional mall is situated at the intersection of Interstate 95 (exit 4A) and Delaware Route 1/Delaware Route 7 (DE 1 exit 164) near the center of the Northeast megalopolis.\nChristiana Mall has three anchor stores - Macy's, JCPenney, and Target - along with Barnes & Noble as a junior anchor and Cabela's and Cinemark Theatres as outparcels. The mall contains 179 shops, and is owned and managed by GGP, a subsidiary of Brookfield Properties. It has 1,267,241 square feet (117,731 m2) of gross leasable area. Christiana Mall is the largest shopping mall in the state of Delaware. The Christiana Mall is located nearly 40 miles (64 km) southwe

In [ ]:
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4g.json", "w", encoding="utf-8") as f:
    json.dump(c4f_pages_db, f, ensure_ascii=False, indent=4)

## Flatten

In [12]:
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"
with open(f"{SAVEFILE_DIR}cleaning_pages_db_4e.json", "r", encoding="utf-8") as f:
    c4g_pages_db = json.load(f)
print("--- Progress Loaded from Checkpoint ---")

--- Progress Loaded from Checkpoint ---


In [13]:
elastic_docs = []

In [14]:
def restructure_page(title, data):

  page_metadata = {
    "last_update": data.get("last_update_text"),
    "full_location": data.get("Full Location"),
    "category": data.get("SE_Category"), # Use Keyword type in ES for filtering
    "location": data.get("SE_Location")   # Use Keyword type in ES for filtering
  }

  # 1. Prepare global data
  summary = data.get("summary", "")
  infobox_raw = data.get("infobox", {})
  # Flatten infobox to string: "Key1: Value1. Key2: Value2."
  infobox_str = ". ".join([f"{k}: {v}" for k, v in infobox_raw.items()])

  # Collect all image URLs for the page to include in the "Lead" doc
  image_urls = [img.get("url") for img in data.get("images", [])]

  # 2. Create the "LEAD" Document (Summary + Infobox)
  lead_doc = {
      "page_title": title,
      "section_title": "Summary",
      "breadcrumb": [title],
      "content": summary,
      "infobox": infobox_str,
      "tables": [], # Summary usually doesn't have tables
      "images": image_urls,
      "is_lead": True,
      **page_metadata
  }
  elastic_docs.append(lead_doc)

  # 3. Pre-process Tables for quick lookup
  # We've mapped them by their hierarchy joined as a string
  table_map = {}
  for table in data.get("table", []):
      # Join hierarchy list into a string key for matching
      path_key = " > ".join(table.get("section_hierarchy", []))
      path_key = title + " > "+path_key
      table_map.setdefault(path_key, []).append({
          "caption": table.get("caption"),
          "rows": table.get("rows")
      })
  #print(table_map)
  # 4. Recursively process the Section Tree
  def walk_tree(sections, parent_breadcrumb):
      #print(f"{sections} - {parent_breadcrumb}")
      for section in sections:
          current_title = section.get("title", "")
          current_breadcrumb = parent_breadcrumb + [current_title]
          breadcrumb_str = " > ".join(current_breadcrumb)
          #print(breadcrumb_str)

          section_text = ""
          subsections = []

          # Handle the "content" field (can be list of text or nested dict)
          raw_content = section.get("content", [])
          if isinstance(raw_content, list):
              for item in raw_content:
                  if isinstance(item, str):
                      section_text += item + " "
                  elif isinstance(item, dict):
                      subsections.append(item)
                  elif isinstance(item, list):
                    for subitem in item:
                      if isinstance(subitem, str):
                          section_text += subitem + " "
                      elif isinstance(subitem, dict):
                          subsections.append(subitem)
          elif isinstance(raw_content, dict):
              # If content itself is a dict, it's a subsection
              subsections.append(raw_content)

          # If this section has text or tables, create a document
          relevant_tables = table_map.get(breadcrumb_str, [])

          if section_text.strip() or relevant_tables:
              elastic_docs.append({
                  "page_title": title,
                  "section_title": current_title,
                  "breadcrumb": current_breadcrumb,
                  "content": section_text.strip(),
                  "infobox": None, # Only lead doc gets infobox
                  "tables": relevant_tables,
                  "images": [], # Subsections usually don't need all images
                  "is_lead": False,
                  **page_metadata
              })

          # Recursively go deeper
          if subsections:
              walk_tree(subsections, current_breadcrumb)

  # Start the recursion for the page
  walk_tree(data.get("sections_tree", []), [title])

In [15]:
elastic_docs = []
title = "Joseph Barboza"
data = c4g_pages_db[title]
restructure_page(title, data)
for i in elastic_docs:
  print(i['section_title'])

Summary
Early life
Professional boxing career
Escape from prison
Entry into organized crime
Turning government witness
False testimony against rivals
Death


In [16]:
len(elastic_docs)

8

In [17]:
elastic_docs

[{'page_title': 'Joseph Barboza',
  'section_title': 'Summary',
  'breadcrumb': ['Joseph Barboza'],
  'content': 'Joseph Barboza Jr. (; September 20, 1932 – February 11, 1976), nicknamed "the Animal", was an American mobster and notorious mob hitman for the Patriarca crime family of New England during the 1960s. A prominent enforcer and contract killer in Boston\'s underworld, Barboza became a Federal Bureau of Investigation (FBI) informant in 1967 and later entered the Witness Protection Program. He was a star witness in the trial of six men convicted in the 1965 murder of Edward Deegan; four of the accused were sentenced to death and another two were sentenced to life imprisonment. It later emerged that Barboza had helped frame the six defendants in a case of wrongful conviction for the Deegan killing, which was allegedly actually committed by Barboza and Vincent Flemmi. He was shot dead in San Francisco in 1976 after his whereabouts became known to Patriarca underboss Gennaro Angiul

In [18]:
elastic_docs = []
for title, data in c4g_pages_db.items():
  restructure_page(title, data)

In [19]:
elastic_docs[402]

{'page_title': 'Battle of Mucojo',
 'section_title': 'Battle',
 'breadcrumb': ['Battle of Mucojo', 'Battle'],
 'content': "On 22 April 2021, militants took and occupied the town of Mucojo and Pangane with the Mozambican military retaking the towns.  The village had been sieged by Mozambican and Rwandan troops since May 2021\nOn 28 August, militants of an unknown group raided and re-occupied the town of Mucojo.  Reports came that the town and a nearby village were captured, with the Mozambique military confirming the claim.  The militants were identified as anti-Islamic as they beheaded an Islamic imam.  The militants continued to kill civilians and behead fisherman.  Members of ISIL and Al-Shabaab, attacked Mucojo and killed most of the anti-Islamic militants in Mucojo and the nearby town of Quiterajo.  The town's mayor was later killed by the Islamist militants with him being beheaded.  The Mozambique army later retook the village, but it was recaptured by the Islamist militants when 

In [20]:
import json

def save_to_jsonl(docs, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')

save_to_jsonl(elastic_docs, f'{SAVEFILE_DIR}flat_wiki_1.jsonl')

### ???